# 从零设计并实现自己的 0.1B LLM
## 一本以理解为中心的 Python / PyTorch 实验教材

你将自己实现一个 **Decoder-only Transformer**，从随机参数开始，依次完成数据与分词器、模型、预训练、SFT、RLVR，最后实现有 KV Cache 的多轮对话。模型不加载任何现成的预训练权重。

**目标模型：98,324,736 个可训练参数，约 0.1B。** `B` 是十亿，`0.1B` 是一亿。它是适合学习的小语言模型；完整训练需要足够的数据和计算，仅运行本教材的短实验不会得到 ChatGPT 的知识与对话质量。

**使用方法：** 选择 Python 内核，从上往下执行。先读文字，预测输出，再运行代码核对。代码中的中文注释解释关键操作；每章都有检查点和小练习。不要一次跳过所有解释直接运行到底。

### 两条运行路线

| 路线 | 模型与数据 | 目的 |
|---|---|---|
| `smoke`，默认 | 相同架构的小模型、自带离线小语料、短训练 | 观察张量、梯度和整个训练流程 |
| `full` | 98.3M 参数、你准备的真实语料、自训 8192 词表 | 在理解后进行 0.1B 模型的实际训练 |

两个配置调用同一套网络代码。小模型用于快速验证思路，完整配置决定真正的参数规模。默认短训练与生成输出都是**教学实验**，不能用来宣称模型已经具备可靠能力。

## 学习地图与完成标准

| 章 | 你要回答的问题 | 完成这一章的证据 |
|---|---|---|
| 0 环境 | Python 与 GPU 如何协作？ | 正确的内核、设备和精度输出 |
| 1 概率与张量 | “预测下一个词”具体预测什么？ | 手算与 PyTorch 交叉熵一致 |
| 2 Tokenizer | 文字怎么变成整数？BPE 合并什么？ | 编码/解码往返一致 |
| 3 数据 | 怎样避免验证集泄漏？ | 清洗去重后分割、token 数与 fingerprint |
| 4 参数设计 | 怎样得到约一亿参数？ | 参数公式与实际统计一致 |
| 5 模型 | Attention、RoPE、SwiGLU 各做什么？ | 逐层 shape、因果性和梯度检查 |
| 6 生成 | 怎么从一次前向变成续写？ | 缓存与全量前向 logits 一致 |
| 7 预训练 | 数据怎样改变参数？ | 短训练、验证损失、保存与恢复 |
| 8 SFT | 为什么只监督 assistant 回答？ | prompt/padding 不计 loss，EOS 计 loss |
| 9 RLVR | 奖励怎样改变回答的概率？ | advantage、policy loss 和真实 rollout |
| 10 评估与聊天 | 怎样区分会背题与会做题？ | 保留题评估、对话与模型导出 |
| 11 走向 0.1B | 长训练需要怎样安排？ | 实际 token 预算、续训方法与学习练习 |

建议分几次学习：第一次到参数设计；第二次手写模型；第三次预训练；第四次 SFT；第五次 RLVR 与评估。你需要会 Python 函数、类和列表；尚不熟悉张量与自动求导时先做第 1 章。

## 0. 准备环境：先让内核使用正确的 Python

本教程已在本地 **PyTorch 2.9.1 + CUDA 13.0、tokenizers 0.22.1** 环境验证。教程依赖安装到你选择的 Notebook 内核；不要因为系统终端里能导入 `torch` 就假设当前内核也可以。

在项目终端执行 `python -m pip install -r requirements.txt`，然后 `python -m jupyter lab`，打开本文件。GPU 版 PyTorch 的安装命令与驱动、平台有关，使用 [PyTorch 官方安装选择器](https://pytorch.org/get-started/locally/) 选择；不要为了本教程重装已经能运行 CUDA 的 PyTorch。

下面这格是本教材的**控制面板**。默认不下载数据、不开启长时间训练。第一次保持 `PROFILE = "smoke"`。准备完整训练时，按第 11 章先放好真实数据，再改成 `full` 并重启内核，从头运行。

In [ ]:
# 基础工具：所有真正的模型、训练与采样代码都将在后续 cell 中展开。
import os, sys, json, math, time, random, hashlib, re, gc, unicodedata
from pathlib import Path
from dataclasses import dataclass, asdict, replace
from collections import Counter
from contextlib import nullcontext
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
import matplotlib.pyplot as plt

# Jupyter 中把图画在 cell 输出区；脚本验证时不创建外部图窗。
if "get_ipython" in globals() and get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

# 可直接改成 "full"；环境变量用于测试脚本或终端启动，普通学习无需设置。
PROFILE = os.environ.get("LLM_PROFILE", "smoke")
assert PROFILE in {"smoke", "full"}
SEED = 42
WORK_DIR = Path(os.environ.get("LLM_ARTIFACT_DIR", f"artifacts/{PROFILE}"))
WORK_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = Path("data/raw")  # full 路线：放自己的 txt/jsonl 语料

# 固定随机种子便于比较实验；CUDA 某些算子仍可能有细小数值差异。
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if DEVICE.type == "cuda" else torch.float32

def amp_context():
    # CPU 保持 FP32；CUDA 在线性运算中使用更省显存的精度。
    return torch.autocast("cuda", dtype=AMP_DTYPE) if DEVICE.type == "cuda" else nullcontext()

print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "设备:", DEVICE, "精度:", AMP_DTYPE)
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "显存 GiB:", round(props.total_memory / 2**30, 2))
print("路线:", PROFILE, "产物目录:", WORK_DIR.resolve())

**观察：** 本机应显示 RTX 4060 Laptop GPU、约 8 GiB 显存和 `cuda`。CPU 也能完成小实验，只是更慢。

**检查点：** 你能解释磁盘上的模型文件、CPU 内存中的参数和 GPU 显存中的张量这三者的区别吗？后面 `x.to(DEVICE)` 是一次设备移动，不是“把 Python 代码搬到 GPU”。

## 1. 先认识模型输出：整数、logits、概率与 loss

假设词表只有三个词：`我、爱、学习`。输入“我 爱”，模型对下一个词输出三个实数 **logits**。Softmax 将它们变成总和为 1 的概率。

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}, \qquad \mathcal L=-\log p_{\text{正确词}}.$$

logits 没有“必须在 0 到 1”这一限制。正确词的概率越低，交叉熵越高。这里先对一个位置手算，再对照 PyTorch。预测：如果把正确词的 logit 调高，loss 会变大还是变小？

In [ ]:
# [B=1, V=3]：只有一个预测位置、三个候选词。
toy_logits = torch.tensor([[0.2, -0.3, 1.1]], requires_grad=True)
toy_target = torch.tensor([2])  # 正确答案为“学习”
toy_probs = toy_logits.softmax(dim=-1)
manual_loss = -toy_probs[0, toy_target.item()].log()
pytorch_loss = F.cross_entropy(toy_logits, toy_target)
torch.testing.assert_close(manual_loss, pytorch_loss)
pytorch_loss.backward()  # 自动求导：告诉我们每个 logit 应怎样调整
print("概率:", toy_probs.detach().tolist(), "loss:", round(pytorch_loss.item(), 4))
print("logits 的梯度:", toy_logits.grad.tolist())
assert toy_logits.grad[0, 2] < 0  # 梯度下降会提高正确词的 logit

### 1.1 一次训练同时预测很多位置

输入 `[我, 爱, 学习]`，目标为 `[爱, 学习, EOS]`。模型在第 0 个位置预测第 1 个 token，最后一个位置预测结束符。**输入和目标相差一位**，这是整个教程最容易写错的地方。

下面的网络只看当前 token，是一个很弱的 bigram 模型。先拿它理解 Embedding、batch、反向传播，第 5 章再学习怎样利用更长的上下文。

In [ ]:
# [B,T] 的整数 -> [B,T,V] 的 logits。Embedding 是可学习的查表，不是分词器。
tiny_bigram = nn.Embedding(4, 4)
bigram_x = torch.tensor([[0, 1, 2]])
bigram_y = torch.tensor([[1, 2, 3]])
bigram_opt = torch.optim.AdamW(tiny_bigram.parameters(), lr=0.15)
bigram_losses = []
for _ in range(30):
    bigram_opt.zero_grad(set_to_none=True)
    logits = tiny_bigram(bigram_x)  # [1,3,4]
    loss = F.cross_entropy(logits.reshape(-1, 4), bigram_y.reshape(-1))
    loss.backward()
    bigram_opt.step()
    bigram_losses.append(loss.item())
print("shape:", logits.shape, "loss:", round(bigram_losses[0], 3), "->", round(bigram_losses[-1], 3))
assert bigram_losses[-1] < bigram_losses[0]

**检查点：** `reshape(-1, V)` 将 batch 与时间合并，每一行是一道“下一 token”分类题。标签必须是 `torch.long`，不是 one-hot 向量。

**练习：** 把训练序列改成 `[0,1,0,2]`。同一个 `0` 有时接 `1`，有时接 `2`，bigram 为什么无法可靠地区分？这个矛盾解释了我们为什么需要能读取上下文的 Transformer。

## 2. Tokenizer：先自己写一个最小 BPE

字符分词简单，但中文、英文、数字需要的序列长度可能差很多。BPE 从很小的符号开始，反复合并最常见的相邻符号，让常见片段成为一个 token。

例子：`low low lower` 中，先统计相邻字符对，可能把 `l,o` 合成 `lo`，再把 `lo,w` 合成 `low`。以下代码只演示**合并规则怎样被学出来**；它不是用于大型语料的高性能分词器，也没有完成特殊 token、字节覆盖和文件持久化。

In [ ]:
def merge_pair(sequence, pair, merged):
    # 从左到右合并不重叠的相邻 pair，例如 a a a -> aa a。
    result, i = [], 0
    while i < len(sequence):
        if i + 1 < len(sequence) and (sequence[i], sequence[i + 1]) == pair:
            result.append(merged)
            i += 2
        else:
            result.append(sequence[i])
            i += 1
    return result

def learn_toy_bpe(text, merges=5):
    # 此处的符号直接用字符串，方便看清每轮合并。
    sequence, rules = list(text), []
    for _ in range(merges):
        pairs = Counter(zip(sequence, sequence[1:]))
        if not pairs:
            break
        pair, count = pairs.most_common(1)[0]
        merged = pair[0] + pair[1]
        sequence = merge_pair(sequence, pair, merged)
        rules.append((pair, merged))
        print("合并:", pair, "次数:", count, "结果:", sequence)
    return sequence, rules

toy_pieces, toy_rules = learn_toy_bpe("low low lower low", merges=4)
assert "".join(toy_pieces) == "low low lower low"

**练习：** 换成“学习 学习 学生”，先手算第一轮。再思考：训练得到的规则在编码新文本时也必须按原顺序应用，为什么不能重新统计新文本的词频？

### 2.1 实际使用 byte-level BPE：先划分数据，再训练分词器

实际分词器用 `tokenizers` 库执行合并算法，但合并规则来自**自己的训练语料**，没有加载外部现成分词器。预先加入完整的 256 个字节符号，让未见过的字符也可以由 UTF-8 字节表示；一个中文字符因此可能对应多个 token。

分词器也是从数据学习的对象，所以先清洗、去重、划分 train/val/test，再只用 train 学规则。为了看清这个顺序，先做第 3 章的数据准备，然后回到本章的实际分词器代码。这种安排避免验证文本参与学习词表。

## 3. 原始数据：小语料用于观察，真实语料用于学习能力

默认自带的文字和算术语料让 Notebook 离线运行。它们数量少、模板重复，验证损失只能检查训练流程，不能证明自然语言泛化。

`full` 路线读取 `data/raw/` 内的文件：`.txt` 每个非空行是一篇文档；`.jsonl` 每行形如 `{"text":"一篇完整文档……"}`。对大文档可以先切成段落；不要把题目与答案拆到两个集合。选择你有权用于训练、语言和领域合适的语料。

数据处理顺序为 **统一格式 → 去除精确重复 → 按文档哈希划分 → 学词表 → 转 token 文件**。这里只做精确去重，近似重复与同源文档应在正式训练前另外分组处理。

In [ ]:
def demo_documents():
    # 这些模板是自写的离线教材数据；形式重复，不能充当真实泛化基准。
    concepts = [
        "语言模型通过预测下一个符号学习文本中的规律。",
        "注意力把相关位置的信息按权重组合起来。",
        "训练集用于更新参数，验证集用于选择训练策略。",
        "梯度下降沿着损失减小的方向调整参数。",
        "Python 的列表可以保存整数，张量可以执行矩阵乘法。",
        "模型先进行预训练，再通过指令微调学习对话格式。",
        "A language model predicts the next token from previous tokens.",
        "Validation data helps us inspect generalization during training.",
    ]
    for i in range(400):
        yield f"教学笔记第{i}页。{concepts[i % len(concepts)]}今天观察第{i % 37}次实验，认真记录输入、输出与损失。"
    for a in range(10):
        for b in range(10):
            yield f"算术练习：{a} + {b} = {a+b}。先计算，再核对最终答案。"

def raw_documents(directory):
    # 流式逐行读取，避免把全部原始文本一次放入内存。
    files = sorted(p for p in directory.rglob("*") if p.suffix.lower() in {".txt", ".jsonl"})
    if not files:
        raise FileNotFoundError(f"请先在 {directory.resolve()} 放 txt/jsonl 真实语料；第一次学习可用 smoke。")
    for path in files:
        with path.open(encoding="utf-8-sig") as handle:
            for line_number, line in enumerate(handle, 1):
                if not line.strip():
                    continue
                if path.suffix.lower() == ".jsonl":
                    record = json.loads(line)
                    if not isinstance(record.get("text"), str):
                        raise ValueError(f"{path}:{line_number} 必须有字符串 text 字段")
                    yield record["text"]
                else:
                    yield line

def clean_document(text):
    # 保留空格与换行语义；不粗暴删掉数字或标点。
    text = unicodedata.normalize("NFC", text).replace("\r\n", "\n").replace("\r", "\n").strip()
    return text if len(text) >= 16 else None

def prepare_splits(documents, directory):
    paths = {name: directory / f"{name}.jsonl" for name in ("train", "val", "test")}
    counts = dict.fromkeys(paths, 0)
    seen = set()  # 内存只存摘要；数千万篇时应换成 SQLite/外部排序。
    handles = {name: path.open("w", encoding="utf-8") for name, path in paths.items()}
    try:
        for raw in documents:
            text = clean_document(raw)
            if text is None:
                continue
            digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
            if digest in seen:
                continue
            seen.add(digest)
            bucket = int(digest[:8], 16) % 10
            split = "train" if bucket < 8 else ("val" if bucket == 8 else "test")
            handles[split].write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
            counts[split] += 1
    finally:
        for handle in handles.values():
            handle.close()
    if min(counts.values()) == 0:
        raise ValueError(f"有空集合 {counts}；请补充文档，不要把验证/测试文本复制进训练集。")
    return paths, counts

split_paths, document_counts = prepare_splits(demo_documents() if PROFILE == "smoke" else raw_documents(RAW_DIR), WORK_DIR)
print("去重后的文档数:", document_counts)

### 3.1 训练自己的分词器，并为对话预留边界

我们使用 6 个特殊符号：padding、BOS（开始）、EOS（结束）、user、assistant、system。它们和普通中文、英文一样对应一个整数 ID，但边界符不能被拆成几个片段。

`8192` 是完整配置的目标词表，不是语料很小时一定能学出的数量。真实语料不足以产生 8192 个符号时，本教材会报错并提示补充数据，避免悄悄改变 0.1B 设计。默认小实验使用较小词表。

这里采用完整字节覆盖、不过滤空格的分词方案，方便验证 `decode(encode(text)) == text`。有关 BPE 的库接口见 [Tokenizers 官方教程](https://huggingface.co/docs/tokenizers/quicktour)。

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

SPECIAL_TOKENS = ["<|pad|>", "<|bos|>", "<|eos|>", "<|user|>", "<|assistant|>", "<|system|>"]

def iter_split_text(path):
    # 可以重复调用迭代器，从磁盘再次读取 train，无需保留全部字符串。
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            yield json.loads(line)["text"]

class TextTokenizer:
    # 轻量包装，让后续训练代码只依赖 encode/decode/vocab_size 三个接口。
    def __init__(self, backend):
        self.backend = backend
        self.vocab_size = backend.get_vocab_size()

    def encode(self, text):
        return self.backend.encode(text, add_special_tokens=False).ids

    def decode(self, ids):
        return self.backend.decode([int(x) for x in ids], skip_special_tokens=True)

target_vocab_size = 1024 if PROFILE == "smoke" else 8192
bpe = Tokenizer(BPE())
# 不自动添加前导空格；标准预切分避免把整篇文档合成一个超长 token。
bpe.pre_tokenizer = ByteLevel(add_prefix_space=False, use_regex=True)
bpe.decoder = ByteLevelDecoder()
bpe.train_from_iterator(
    iter_split_text(split_paths["train"]),
    trainer=BpeTrainer(vocab_size=target_vocab_size, min_frequency=2,
                       special_tokens=SPECIAL_TOKENS, initial_alphabet=ByteLevel.alphabet()),
)
tokenizer = TextTokenizer(bpe)
if PROFILE == "full" and tokenizer.vocab_size != 8192:
    raise ValueError(f"只学出 {tokenizer.vocab_size} 个 token；请增加真实训练语料后再训练 8192 词表。")
bpe.save(str(WORK_DIR / "tokenizer.json"))
PAD_ID, BOS_ID, EOS_ID, USER_ID, ASSISTANT_ID, SYSTEM_ID = [bpe.token_to_id(s) for s in SPECIAL_TOKENS]
probe_text = "你好，LLM! 12 + 13 = 25 🐱\n学习 Python。"
probe_ids = tokenizer.encode(probe_text)
assert tokenizer.decode(probe_ids) == probe_text
assert all(len(tokenizer.encode(s)) == 1 for s in SPECIAL_TOKENS)
print("实际词表:", tokenizer.vocab_size, "特殊ID:", [PAD_ID,BOS_ID,EOS_ID,USER_ID,ASSISTANT_ID,SYSTEM_ID])
print("编码:", probe_ids, "\n解码:", tokenizer.decode(probe_ids))

**检查点：** 分词器的 ID 不能在预训练后随意改变。换词表会让同一个整数代表另一个词，而 Embedding 仍保留旧含义。后续 checkpoint 会记录分词器摘要，恢复时核对。

**练习：** 比较中文句子、英文句子和连续数字的 token 数。词表扩大可能缩短序列，但也增加 Embedding 与 LM Head 的开销；第 4 章会把这项代价写成公式。

### 3.2 从文档变成训练用的 token 流

把文档编码后接上 EOS，再顺序写入二进制文件。`memmap` 让我们按需读取 token，避免把长语料全部变成 GPU 张量。只在取一个 batch 时才转 `torch.long` 并搬到设备。

不同文档之间用 EOS 分隔。本入门实现允许文档之间做因果注意力，EOS 是边界提示；大型训练还可使用文档级注意力掩码。训练、验证、测试始终写在三个不同文件中。

In [ ]:
def file_sha256(path):
    # 分块哈希，不把整个 token 文件读进内存。
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_token_stream(jsonl_path, binary_path):
    count = 0
    with binary_path.open("wb") as output:
        for text in iter_split_text(jsonl_path):
            ids = tokenizer.encode(text) + [EOS_ID]
            assert max(ids) < 65536  # uint16 的范围，本教程词表小于此值
            np.asarray(ids, dtype="<u2").tofile(output)
            count += len(ids)
    return count

token_counts = {}
manifest = {"profile": PROFILE, "documents": document_counts,
            "tokenizer_sha256": file_sha256(WORK_DIR / "tokenizer.json"), "splits": {}}
for split, path in split_paths.items():
    output_path = WORK_DIR / f"{split}.bin"
    token_counts[split] = write_token_stream(path, output_path)
    manifest["splits"][split] = {"tokens": token_counts[split], "sha256": file_sha256(output_path)}
DATA_FINGERPRINT = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
manifest["fingerprint"] = DATA_FINGERPRINT
(WORK_DIR / "data_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
train_tokens, val_tokens, test_tokens = [np.memmap(WORK_DIR / f"{s}.bin", dtype="<u2", mode="r") for s in ("train", "val", "test")]
print("token 数:", token_counts, "\nfingerprint:", DATA_FINGERPRINT[:16])
print("一段训练文本:", tokenizer.decode(train_tokens[:80]))

## 4. 先算参数，再设计 0.1B 模型

设词表大小 $V$，隐藏维度 $d$，层数 $L$，SwiGLU 中间维度 $m$。使用无 bias 的线性层、每层两个 RMSNorm、Embedding 与输出权重共享时：

$$N = Vd + L(4d^2 + 3dm + 2d) + d.$$

- $Vd$：Embedding 与 LM Head 共享同一张参数表，因此只计一次。
- $4d^2$：Q、K、V、输出投影，四个 $d\times d$ 矩阵。
- $3dm$：SwiGLU 的两个升维矩阵和一个降维矩阵。
- $2d$：每个 block 两个 RMSNorm 的缩放向量。
- 最后的 $d$：输出前的 RMSNorm。RoPE 使用固定函数，没有可训练位置参数。

我们选 **$V=8192,d=768,L=13,m=2048$**，得到 **98,324,736**。12 个注意力头使每头维度为 64；头数本身不直接增加参数，但改变信息处理和计算布局。13 层虽然不是常见整数倍，架构上完全合法。

为什么选 $m=2048$？普通两层 FFN 若中间维度为 $4d$，参数是 $8d^2$；SwiGLU 有三张矩阵，令 $3dm=8d^2$，得到 $m=8d/3=2048$。这样在引入门控时保持类似的参数预算。

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int = 8192
    dim: int = 768
    n_layers: int = 13
    n_heads: int = 12
    hidden_dim: int = 2048
    max_seq_len: int = 256
    use_sdpa: bool = True
    gradient_checkpointing: bool = True

    def __post_init__(self):
        assert self.dim % self.n_heads == 0, "hidden dim 必须能均分成注意力头"
        assert (self.dim // self.n_heads) % 2 == 0, "RoPE 每头维度必须为偶数"
        assert min(self.vocab_size,self.dim,self.n_layers,self.n_heads,self.hidden_dim,self.max_seq_len) > 0

def parameter_budget(config):
    V, d, L, m = config.vocab_size, config.dim, config.n_layers, config.hidden_dim
    return V*d + L*(4*d*d + 3*d*m + 2*d) + d

FULL_CFG = ModelConfig()
cfg = FULL_CFG if PROFILE == "full" else ModelConfig(
    vocab_size=tokenizer.vocab_size, dim=64, n_layers=2, n_heads=4,
    hidden_dim=176, max_seq_len=64, use_sdpa=True, gradient_checkpointing=False,
)
assert parameter_budget(FULL_CFG) == 98_324_736
print("完整目标:", f"{parameter_budget(FULL_CFG):,}", "参数")
print("本次路线:", asdict(cfg), "\n本次参数:", f"{parameter_budget(cfg):,}")

### 4.1 一亿参数在 8 GiB 显存里占多少？

采用 FP32 参数 + AdamW 时，参数、梯度、两份优化器动量约为 **$16N$ 字节**。98.3M 参数约占 1.47 GiB。这还没算激活值、logits、CUDA 工作区、显存碎片、RL 的 reference 副本等。

所以“参数能放下”不等于“任意 batch 都能训练”。本机从 micro batch 1、context 256、AMP、激活重计算开始。梯度累计增加有效 batch，通常不增加单个 micro batch 的激活峰值。先跑短步测显存，再调整。

注意：AMP 是按算子混用精度，本教程的模型参数仍为 FP32；不要在 AMP 训练前再随意 `model.half()`。参考 [PyTorch 2.9 AMP 文档](https://docs.pytorch.org/docs/2.9/amp.html)。

In [ ]:
# 这是静态估算，不是承诺的实际峰值；真实峰值在第 11 章测量。
N = parameter_budget(FULL_CFG)
print("仅 FP32 参数 MiB:", round(N*4 / 2**20, 1))
print("参数+梯度+AdamW 状态 GiB:", round(N*16 / 2**30, 2))
print("batch=1,context=256 的 FP32 logits MiB:", round(1*256*8192*4 / 2**20, 1))

**练习：** 只改词表到 16384，算参数增加多少；只把 context 改到 512，为什么参数量不变但训练显存会上升？写下答案，再继续手写网络。

## 5. 手写 Transformer：先看数据怎样流动

以下维度贯穿本章：$B$ 是 batch，$T$ 是 token 数，$d$ 是隐藏维度，$H$ 是头数，$d_h=d/H$ 是每头维度，$V$ 是词表大小。

```text
整数 ID [B,T]
  ↓ Embedding：查表
隐藏状态 [B,T,d]
  ↓ 重复 L 次 Transformer Block
  ├─ RMSNorm → Causal Attention（Q、K 上施加 RoPE）→ 加回残差
  └─ RMSNorm → SwiGLU → 加回残差
  ↓ 最后的 RMSNorm
  ↓ 共享 Embedding 的线性输出层
logits [B,T,V]
```

一个 block 改变表示，保持 `[B,T,d]` 形状。这样才能堆叠多层。不要一次复制整套模型；先实现每个部件，并在小张量上检查它。

### 5.1 RMSNorm：稳定每个 token 的特征尺度

对一个 token 的 $d$ 维向量 $x$：

$$\mathrm{RMSNorm}(x)_i = \frac{x_i}{\sqrt{\frac1d\sum_j x_j^2+\epsilon}}\,g_i.$$

$g$ 是可学习的缩放向量，初值为 1。它沿**最后的特征维度**归一化，不会混合不同 token。与 LayerNorm 不同，它不减均值。平方与求均值在 FP32 做，以减少低精度数值问题，再转回输入类型。

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        original_dtype = x.dtype
        x_float = x.float()
        inv_rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * inv_rms * self.weight.float()).to(original_dtype)

norm_probe = RMSNorm(4)
norm_input = torch.tensor([[[1., 2., 3., 4.], [2., 4., 6., 8.]]])
norm_output = norm_probe(norm_input)
print("输入/输出:", norm_input.shape, norm_output.shape)
print("归一化后的均方根:", norm_output.square().mean(-1).sqrt().detach())
torch.testing.assert_close(norm_output[0,0], norm_output[0,1], rtol=1e-5, atol=1e-5)

**观察：** 两个 token 只差一个正比例倍数，归一化后近似相同。输出 shape 不变。

### 5.2 位置从哪里来？RoPE 把 Q、K 旋转

没有位置信息时，注意力很难区分“我爱你”和“你爱我”。RoPE 将每个注意力头的偶数、奇数坐标两两配对，以随位置变化的角度旋转：

$$\begin{bmatrix}x'_{2i}\\x'_{2i+1}\end{bmatrix}
=\begin{bmatrix}\cos\theta&-\sin\theta\\\sin\theta&\cos\theta\end{bmatrix}
\begin{bmatrix}x_{2i}\\x_{2i+1}\end{bmatrix},\quad
\theta=\mathrm{position}\times10000^{-2i/d_h}.$$

旋转保持向量长度；旋转后的 Q、K 点积会携带相对位置信息。我们对 Q、K 使用 RoPE，V 保留原值。缓存解码时必须从历史 token 数开始计位置，不能每次都从 0 开始。参见 [RoFormer 原论文](https://arxiv.org/abs/2104.09864)。

In [ ]:
def apply_rope(x, positions):
    # x: [B,H,T,Dh]；positions: [T]，可以从非零位置开始。
    head_dim = x.shape[-1]
    inv_freq = 10000.0 ** (-torch.arange(0, head_dim, 2, device=x.device).float() / head_dim)
    angles = positions.float()[:, None] * inv_freq[None, :]
    cos = angles.cos()[None, None, :, :].to(x.dtype)
    sin = angles.sin()[None, None, :, :].to(x.dtype)
    even, odd = x[..., 0::2], x[..., 1::2]
    rotated = torch.stack((even*cos - odd*sin, even*sin + odd*cos), dim=-1)
    return rotated.flatten(-2)  # 把每对坐标恢复为一个 Dh 维向量

rope_probe = torch.randn(1, 2, 4, 8)
rope_result = apply_rope(rope_probe, torch.arange(4))
torch.testing.assert_close(rope_probe.square().sum(-1), rope_result.square().sum(-1), rtol=1e-5, atol=1e-5)
torch.testing.assert_close(rope_probe[:,:,0], rope_result[:,:,0])
print("位置 0 保持不变；其他位置改变方向但保持长度。")

**练习：** 把所有 `positions` 改成 0，预测结果；再把位置从 10 开始，验证长度仍不变。

### 5.3 Attention：每个位置提出问题，历史位置提供信息

$Q=xW_Q$ 表示“我正在寻找什么”，$K=xW_K$ 表示“我能被怎样匹配”，$V=xW_V$ 表示“我提供什么信息”。这些描述是帮助理解的比喻，三个矩阵都由训练学习。

$$A=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+M\right),\qquad O=AV.$$

**因果掩码 $M$** 在未来位置为 $-\infty$，其他位置为 0。Softmax 后未来权重为 0。若忘记掩码，模型训练时可以偷看答案，loss 会好看，生成却失败。

In [ ]:
# 先用一个头、三个位置观察注意力矩阵，不涉及网络类。
q_demo, k_demo, v_demo = [torch.randn(1,1,3,4) for _ in range(3)]
allowed_demo = torch.ones(3,3,dtype=torch.bool).tril()
scores_demo = q_demo @ k_demo.transpose(-2,-1) / math.sqrt(4)
weights_demo = scores_demo.masked_fill(~allowed_demo, float("-inf")).softmax(-1)
output_demo = weights_demo @ v_demo
print("每行对应一个查询位置，每列对应一个被读取的位置:")
print(weights_demo[0,0])
assert torch.equal(weights_demo[0,0].triu(1), torch.zeros(3,3))
torch.testing.assert_close(weights_demo.sum(-1), torch.ones(1,1,3))
print("输出:", output_demo.shape)

### 5.4 多头与实际实现

多头只是把 $d$ 维拆成 $H$ 组，每组做独立注意力，再拼回来。下面的 `manual` 路径正好实现上面的公式；`SDPA` 路径让 PyTorch 选择更省显存的实现，**没有调用预训练 Transformer**。

缓存 `K,V` 的长度为 $P$，新输入长度为 $T$ 时，查询位置为 $P,P+1,\ldots$，键位置为 $0,1,\ldots,P+T-1$。合法条件是 `key_position <= query_position`。缓存场景不可以简单对一个非方阵套左上角三角掩码。

PyTorch SDPA 的布尔 mask 中，`True` 表示允许注意。我们把 `dropout_p` 固定为 0，便于比较路径与检查缓存；`is_causal=True` 只用于没有历史的方阵。接口和 mask 语义见 [PyTorch 2.9 SDPA 文档](https://docs.pytorch.org/docs/2.9/generated/torch.nn.functional.scaled_dot_product_attention.html)。

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.head_dim = config.dim // config.n_heads
        self.use_sdpa = config.use_sdpa
        self.qkv = nn.Linear(config.dim, 3*config.dim, bias=False)
        self.out_proj = nn.Linear(config.dim, config.dim, bias=False)

    def forward(self, x, kv_cache=None, use_cache=False):
        batch, length, dim = x.shape
        # 一次投影得到 Q/K/V，然后拆出头维度。
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(batch,length,self.n_heads,self.head_dim).transpose(1,2) for t in (q,k,v)]
        past_length = 0 if kv_cache is None else kv_cache[0].shape[-2]
        positions = torch.arange(past_length, past_length+length, device=x.device)
        q, k = apply_rope(q,positions), apply_rope(k,positions)
        if kv_cache is not None:
            k = torch.cat((kv_cache[0],k), dim=-2)
            v = torch.cat((kv_cache[1],v), dim=-2)
        key_positions = torch.arange(k.shape[-2], device=x.device)
        allowed = key_positions[None,:] <= positions[:,None]
        if self.use_sdpa:
            if past_length == 0:
                out = F.scaled_dot_product_attention(q,k,v,dropout_p=0.0,is_causal=True)
            else:
                out = F.scaled_dot_product_attention(q,k,v,attn_mask=allowed,dropout_p=0.0)
        else:
            scores = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
            # Softmax 在 FP32 计算，之后转回 V 的 dtype。
            weights = scores.float().masked_fill(~allowed,float("-inf")).softmax(-1).to(v.dtype)
            out = weights @ v
        out = out.transpose(1,2).contiguous().view(batch,length,dim)
        new_cache = (k,v) if use_cache else None
        return self.out_proj(out), new_cache

attention_probe = CausalAttention(replace(cfg,use_sdpa=False))
attention_x = torch.randn(1,4,cfg.dim)
attention_y, _ = attention_probe(attention_x)
print("多头前后:", attention_x.shape, "->", attention_y.shape)

**检查点：** Q/K/V shape 是 `[B,H,T,Dh]`；注意力权重是 `[B,H,T,T]`；合并后又是 `[B,T,d]`。不要把 `H` 与 `T` 转置反了。

### 5.5 SwiGLU 与残差：既做信息交换，也做特征变换

Attention 在位置之间交换信息；FFN 则独立处理每个位置的特征。SwiGLU 使用一个分支作为门：

$$\mathrm{FFN}(x)=\big(\mathrm{SiLU}(xW_g)\odot(xW_u)\big)W_d,$$

其中两个升维投影得到 `[B,T,m]`，逐元素相乘，再降回 `[B,T,d]`。残差加法 `x + 分支(x)` 让梯度有更直接的通路。我们用 Pre-Norm：先归一化再进入分支。

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate = nn.Linear(config.dim, config.hidden_dim, bias=False)
        self.up = nn.Linear(config.dim, config.hidden_dim, bias=False)
        self.down = nn.Linear(config.hidden_dim, config.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = RMSNorm(config.dim)
        self.attn = CausalAttention(config)
        self.ffn_norm = RMSNorm(config.dim)
        self.ffn = SwiGLU(config)

    def forward(self, x, kv_cache=None, use_cache=False):
        attention_output, new_cache = self.attn(self.attn_norm(x),kv_cache,use_cache)
        x = x + attention_output  # 第一个残差
        x = x + self.ffn(self.ffn_norm(x))  # 第二个残差
        return x, new_cache

block_probe = TransformerBlock(cfg)
block_y, _ = block_probe(torch.randn(1,4,cfg.dim))
assert block_y.shape == (1,4,cfg.dim)
print("一个 Transformer Block 保持形状:", block_y.shape)

### 5.6 把部件组装成语言模型

最后一层输出 $h$ 后，乘 Embedding 参数的转置得到词表 logits：$z=hE^\top$。同一个词向量既负责读 token，也负责与输出特征匹配，这就是权重共享。

完整配置会开启**激活重计算**：反向传播时重新执行 block 前向，减少训练中保存的激活，代价是额外计算。只有训练且需要梯度时启用；推理缓存不走这条路径。它与“保存模型的 checkpoint 文件”是两件事。

In [ ]:
class TinyLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.cfg = config
        self.embedding = nn.Embedding(config.vocab_size,config.dim)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.final_norm = RMSNorm(config.dim)
        self.lm_head = nn.Linear(config.dim,config.vocab_size,bias=False)
        self.apply(self._init_weights)
        self.lm_head.weight = self.embedding.weight  # 同一个 Parameter，仅统计与优化一次

    def _init_weights(self, module):
        if isinstance(module,(nn.Linear,nn.Embedding)):
            nn.init.normal_(module.weight,mean=0.0,std=0.02)

    def forward(self, ids, use_cache=False, kv_cache=None):
        if ids.ndim != 2 or ids.shape[1] == 0:
            raise ValueError("输入应为非空 [batch,time] 整数 token")
        if kv_cache is not None and len(kv_cache) != len(self.blocks):
            raise ValueError("缓存层数与模型层数不符")
        past = 0 if kv_cache is None else kv_cache[0][0].shape[-2]
        if past + ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("超出上下文长度；聊天时请先裁剪完整历史轮次")
        x, new_cache = self.embedding(ids), []
        for index, block in enumerate(self.blocks):
            layer_cache = None if kv_cache is None else kv_cache[index]
            if self.cfg.gradient_checkpointing and self.training and torch.is_grad_enabled() and not use_cache and kv_cache is None:
                # 直接传当前 block 的绑定方法，避免循环闭包在反向时引用最后一层。
                x, _ = checkpoint(block,x,use_reentrant=False)
                cache = None
            else:
                x, cache = block(x,layer_cache,use_cache)
            if use_cache:
                new_cache.append(cache)
        logits = self.lm_head(self.final_norm(x))
        return logits, (new_cache if use_cache else None)

model = TinyLLM(cfg).to(DEVICE)
actual_parameters = sum(p.numel() for p in model.parameters())
assert actual_parameters == parameter_budget(cfg)
assert model.lm_head.weight is model.embedding.weight
print("实际可训练参数:", f"{actual_parameters:,}", "公式一致；输出权重共享。")

### 5.7 最关键的模型自检：看不到未来，梯度能传回去

先拿两条前缀相同、后缀不同的输入。由于因果掩码，公共前缀位置的 logits 应相同。然后做一次反向传播，检查 loss 和梯度都是有限数。

这一步比“模型运行没有报错”更有意义：一个忘了因果掩码的模型同样能运行。

In [ ]:
model.eval()
ids_a = torch.tensor([[BOS_ID]+tokenizer.encode("学习语言模型")[:12]],device=DEVICE)
ids_b = ids_a.clone()
prefix_length = max(1,ids_a.shape[1]//2)
ids_b[:,prefix_length:] = (ids_b[:,prefix_length:]+1) % cfg.vocab_size
with torch.no_grad():
    logits_a, _ = model(ids_a)
    logits_b, _ = model(ids_b)
torch.testing.assert_close(logits_a[:,:prefix_length],logits_b[:,:prefix_length],rtol=2e-4,atol=2e-5)

# 所有位置都预测下一个 token：此时 x 和 y 已经错开，不再重复 shift。
model.train()
ids_probe = torch.tensor(np.array(train_tokens[:17],dtype=np.int64),device=DEVICE).unsqueeze(0)
logits_probe, _ = model(ids_probe[:,:-1])
loss_probe = F.cross_entropy(logits_probe.reshape(-1,cfg.vocab_size),ids_probe[:,1:].reshape(-1))
loss_probe.backward()
assert torch.isfinite(loss_probe)
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
print("因果性通过；loss:", round(loss_probe.item(),4), "Embedding梯度范数:", model.embedding.weight.grad.norm().item())
model.zero_grad(set_to_none=True)  # 自检不是一次训练，清理其梯度

**练习：** 暂时把手写 Attention 的 mask 去掉，再跑公共前缀测试，观察失败。恢复 mask 后再继续。你的测试现在能捕捉一种真实的模型错误。

## 6. 自回归生成与 KV Cache

训练可并行预测很多位置；生成必须一步一步：读 prompt → 看最后一个位置的 logits → 选择一个 token → 接回输入 → 再预测。

缓存保留每层历史 token 的 K、V。新 token 只算自己的 Q/K/V，然后读取旧 K/V；历史 token 的隐藏状态不需要每次重算。这不改变模型的预测含义，但减少重复工作。

In [ ]:
model.eval()
cache_ids = ids_probe[:,:12]
with torch.no_grad():
    full_logits, _ = model(cache_ids)
    first_logits, cache = model(cache_ids[:,:8],use_cache=True)
    # 一次追加4个token，检查非方阵因果mask与RoPE位置偏移。
    next_logits, cache = model(cache_ids[:,8:],use_cache=True,kv_cache=cache)
    cached_logits = torch.cat((first_logits,next_logits),dim=1)
torch.testing.assert_close(full_logits,cached_logits,rtol=2e-4,atol=2e-5)
print("缓存与全量 logits 一致；每层缓存 K shape:", cache[0][0].shape)

### 6.1 怎样选下一个 token？

- 贪心：取最高 logit，方便可重复的评估。
- Temperature：logits 除以正数温度；小于 1 更集中，大于 1 更分散。
- Top-k：只保留概率最高的 k 个候选。
- Top-p：保留累计概率刚达到 p 的最小候选前缀。

Top-k/top-p 改变采样分布。聊天可以用它们，后面 RL 为了正确计算策略概率，将固定温度 1 且不截断。EOS 停止生成；`max_new_tokens` 防止永不结束。

In [ ]:
def sample_next_token(logits, temperature=1.0, top_k=None, top_p=1.0):
    # logits 为 [1,V]；返回 [1,1]，与输入的 batch 维保持一致。
    if temperature < 0 or not 0 < top_p <= 1:
        raise ValueError("temperature >= 0，且 0 < top_p <= 1")
    if temperature == 0:
        return logits.argmax(dim=-1,keepdim=True)
    scores = logits.float() / temperature
    if top_k is not None:
        if not isinstance(top_k,int) or top_k < 1:
            raise ValueError("top_k 必须为正整数或 None")
        k = min(top_k,scores.shape[-1])
        cutoff = scores.topk(k,dim=-1).values[:,-1,None]
        scores = scores.masked_fill(scores < cutoff,float("-inf"))
    if top_p < 1:
        sorted_scores, order = scores.sort(dim=-1,descending=True)
        cumulative = sorted_scores.softmax(-1).cumsum(-1)
        remove = cumulative > top_p
        remove[:,1:] = remove[:,:-1].clone()  # 保留首次超过阈值的那个 token
        remove[:,0] = False
        sorted_scores = sorted_scores.masked_fill(remove,float("-inf"))
        scores = torch.full_like(scores,float("-inf")).scatter(-1,order,sorted_scores)
    return torch.multinomial(scores.softmax(-1),num_samples=1)

@torch.no_grad()
def generate(policy, prompt_ids, max_new_tokens=32, temperature=1.0, top_k=None, top_p=1.0, use_cache=True):
    if not prompt_ids or max_new_tokens < 0:
        raise ValueError("prompt 不为空；max_new_tokens >= 0")
    if len(prompt_ids)+max_new_tokens > policy.cfg.max_seq_len:
        raise ValueError("prompt + 生成预算超出上下文；请缩短输入或生成预算")
    was_training = policy.training
    policy.eval()
    device = next(policy.parameters()).device
    ids = torch.tensor([prompt_ids],dtype=torch.long,device=device)
    result, cache = list(prompt_ids), None
    try:
        for _ in range(max_new_tokens):
            # 第一步读整个 prompt，之后缓存路径只读刚生成的 token。
            with amp_context():
                logits, cache = policy(ids,use_cache=use_cache,kv_cache=cache)
            token = sample_next_token(logits[:,-1],temperature,top_k,top_p)
            result.append(int(token.item()))
            if result[-1] == EOS_ID:
                break
            ids = token if use_cache else torch.tensor([result],dtype=torch.long,device=device)
    finally:
        policy.train(was_training)
    return result

untrained_prompt = [BOS_ID]+tokenizer.encode("语言模型")
untrained_output = generate(model,untrained_prompt,max_new_tokens=8,temperature=0)
print("尚未训练时的续写:", repr(tokenizer.decode(untrained_output)))

**观察：** 随机参数的输出可能是乱码、重复符号或不通顺的词片段，这很正常。架构本身不会自带中文知识。

**练习：** 设 `top_k=1` 与贪心相比，结果应该怎样？如果模型立刻输出 EOS，空回答能否直接证明生成代码有错？需要结合训练与 logits 分析。

到这里你已经实现了一个可以前向、反向、带缓存生成的完整语言模型。接下来，学习数据如何把这个概率模型训练成基座模型，再训练成指令模型。

## 7. 预训练：把“预测下一个 token”变成一次参数更新

前面我们已经得到分词器、三份 token 数据和 `TinyLLM`。这一章只解决一个问题：**怎样用文本教模型预测下一个 token？**

学习顺序是：移位样本 → 有效 token 损失 → 学习率 → 验证 → 保存与恢复 → 短训练。运行默认的 20 步 smoke 只验证流程；它不代表模型已经学会语言。约 0.1B 参数的正式配置也需要足够的高质量数据和训练 token。

本章约定：`steps` 始终表示成功执行的 **optimizer updates**；`grad_accum` 个小批次才组成一次更新。输入和目标的形状均为 `[B,T]`，模型输出为 `[B,T,V]`。

### 7.1 为什么目标要右移一位？

给定窗口 `[我, 喜欢, 学习, EOS]`，输入是 `[我, 喜欢, 学习]`，目标是 `[喜欢, 学习, EOS]`。因果掩码使位置 `t` 只能看见 `0..t`，目标是 `t+1`，因此不会偷看答案。

\[
\mathcal L=-\frac{1}{N}\sum_{b,t\in\text{有效位置}}\log p_\theta(y_{b,t}\mid x_{b,0:t}).
\]

`TokenWindowDataset` 把索引解释为窗口起点；随机性由 sampler 提供。它只复制当前长度 `T+1` 的窗口，适合读取磁盘上的 `np.memmap`，不用把整份语料搬进内存。

In [ ]:
class TokenWindowDataset(Dataset):
    def __init__(self, tokens, seq_len):
        # 至少需要 T+1 个 token，最后一个 token 只出现在目标中。
        self.tokens = tokens
        self.seq_len = int(seq_len)
        if self.seq_len < 1 or len(tokens) <= self.seq_len:
            raise ValueError("token 数据必须至少包含 seq_len+1 个元素")

    def __len__(self):
        return len(self.tokens) - self.seq_len

    def __getitem__(self, start):
        start = int(start)
        if not 0 <= start < len(self):
            raise IndexError(start)
        # 显式复制只读 memmap 切片，转换成 Embedding 需要的 int64。
        window = torch.from_numpy(
            np.array(self.tokens[start:start + self.seq_len + 1], dtype=np.int64)
        )
        return window[:-1], window[1:]

# 可观察自检：同一份窗口在输入、目标中只错开一个位置。
toy_window = TokenWindowDataset(np.arange(9), seq_len=4)
toy_x, toy_y = toy_window[2]
assert toy_x.tolist() == [2, 3, 4, 5]
assert toy_y.tolist() == [3, 4, 5, 6]
print("输入：", toy_x.tolist(), "目标：", toy_y.tolist())

### 7.2 随机训练窗口与固定验证窗口

训练窗口可以重复采样，这是从长 token 流随机截取片段的简单办法。验证采用固定、均匀分布的起点，便于重复比较；验证文本必须来自先前隔离的 `val_tokens`，不能从训练 token 流中截取。

显存由**小批大小**和 `T` 影响，而有效批量还取决于梯度累计。若 `B=1,T=256,K=16`，一次更新处理约 `4096` 个目标 token。Windows 的 Notebook 使用 `num_workers=0`，避免工作进程导入 Notebook 状态的问题。

In [ ]:
from torch.utils.data import RandomSampler, Subset

PT_BATCH_SIZE = int(os.environ.get("LLM_PT_BATCH_SIZE", "4" if PROFILE == "smoke" else "1"))
PT_GRAD_ACCUM = int(os.environ.get("LLM_PT_GRAD_ACCUM", "2" if PROFILE == "smoke" else "16"))
pretrain_dataset = TokenWindowDataset(train_tokens, cfg.max_seq_len)
pretrain_sampler_generator = torch.Generator().manual_seed(2026)
pretrain_sampler = RandomSampler(
    pretrain_dataset, replacement=True,
    num_samples=min(len(pretrain_dataset), 8192),
    generator=pretrain_sampler_generator,
)
pretrain_loader = DataLoader(
    pretrain_dataset, batch_size=PT_BATCH_SIZE, sampler=pretrain_sampler,
    num_workers=0, pin_memory=DEVICE.type == "cuda",
)

# 验证起点固定；按 token 数加权的损失函数会在下一节实现。
val_dataset = TokenWindowDataset(val_tokens, cfg.max_seq_len)
val_count = min(len(val_dataset), 128 if PROFILE == "smoke" else 1024)
val_starts = np.linspace(0, len(val_dataset) - 1, val_count, dtype=np.int64).tolist()
pretrain_val_loader = DataLoader(
    Subset(val_dataset, val_starts), batch_size=PT_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=DEVICE.type == "cuda",
)
batch_x, batch_y = next(iter(pretrain_loader))
assert batch_x.shape == batch_y.shape
assert torch.equal(batch_x[:, 1:], batch_y[:, :-1])
print("小批形状：", tuple(batch_x.shape), "每次更新约含 token：",
      PT_BATCH_SIZE * cfg.max_seq_len * PT_GRAD_ACCUM)

### 7.3 损失和 AdamW 各自负责什么？

交叉熵把 logits `[B,T,V]` 展开成 `[B*T,V]`，目标展开成 `[B*T]`。`-100` 表示不参与损失的目标；预训练窗口没有 padding，但这个约定会贯穿 SFT。

我们先求损失之和与有效 token 数，再由调用者做除法。这样长短不等的小批不会获得相同权重。AdamW 更新参数，并对矩阵权重使用 weight decay；偏置、归一化向量和其他一维参数不衰减。

In [ ]:
def token_loss_sum(logits, targets):
    # targets 已由 Dataset 移位；这里绝对不能再移位一次。
    total = F.cross_entropy(
        logits.float().reshape(-1, logits.size(-1)), targets.reshape(-1),
        ignore_index=-100, reduction="sum",
    )
    count = int((targets != -100).sum().item())
    return total, count


def make_optimizer(model, lr):
    matrices, vectors = [], []
    for parameter in model.parameters():
        if parameter.requires_grad:
            (matrices if parameter.ndim >= 2 else vectors).append(parameter)
    return torch.optim.AdamW(
        [{"params": matrices, "weight_decay": 0.1},
         {"params": vectors, "weight_decay": 0.0}],
        lr=lr, betas=(0.9, 0.95), eps=1e-8,
    )

# 可观察自检：均匀分布预测的交叉熵应当是 log(V)。
uniform_logits = torch.zeros(2, 3, 7, requires_grad=True)
masked_targets = torch.tensor([[1, 2, -100], [0, -100, 4]])
uniform_sum, uniform_count = token_loss_sum(uniform_logits, masked_targets)
assert uniform_count == 4
assert abs(uniform_sum.item() / uniform_count - math.log(7)) < 1e-6
print("有效目标：", uniform_count, "平均损失：", uniform_sum.item() / uniform_count)

### 7.4 为什么先 warmup，再慢慢降低学习率？

刚开始时参数和 Adam 的统计量尚不稳定，先从小学习率增加到峰值；随后用余弦曲线降低步长。以下 `update_index` 从 `0` 开始计数，代表下一次 optimizer update。

\[
\eta(s)=\eta_{\max}\frac{s+1}{W}\quad(s<W),\qquad
\eta(s)=\eta_{\min}+\frac{\eta_{\max}-\eta_{\min}}{2}(1+\cos(\pi q)).
\]

其中 `q` 是 warmup 之后的归一化进度。短演示的默认 warmup 占 10%；正式训练时应根据总更新数设置。它是教学默认值，不是对所有语料都最优的超参数。

In [ ]:
def scheduled_lr(update_index, total_steps, peak_lr, warmup_steps, min_ratio=0.1):
    if update_index < warmup_steps:
        return peak_lr * (update_index + 1) / max(1, warmup_steps)
    progress = (update_index - warmup_steps) / max(1, total_steps - warmup_steps - 1)
    progress = min(1.0, max(0.0, progress))
    return peak_lr * (min_ratio + (1 - min_ratio) * (1 + math.cos(math.pi * progress)) / 2)

lr_preview = [scheduled_lr(s, 20, 3e-4, 2) for s in range(20)]
assert abs(lr_preview[1] - 3e-4) < 1e-12
assert abs(lr_preview[-1] - 3e-5) < 1e-12
plt.figure(figsize=(6, 2.5))
plt.plot(range(1, 21), lr_preview, marker=".")
plt.xlabel("optimizer update")
plt.ylabel("learning rate")
plt.grid(alpha=0.3)
plt.show()

### 7.5 验证损失怎样公平地平均？

假设两个验证批分别包含 `4` 和 `1` 个有效 token，平均损失分别为 `2` 和 `8`。正确总体平均是 `(4*2+1*8)/5=3.2`，不是 `(2+8)/2=5`。

验证不更新参数；暂时切换为 `eval`，结束后恢复原模式。困惑度为 `exp(loss)`，仅在同一分词器、同一验证集和相同掩码约定下可比较。`max_batches` 可让教学试跑更快，但限制后的指标只代表所评估的子集。

In [ ]:
@torch.no_grad()
def evaluate_lm(model, loader, max_batches=None):
    was_training = model.training
    model.eval()
    total_loss, total_tokens = 0.0, 0
    try:
        for batch_index, (inputs, targets) in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            with amp_context():
                logits, _ = model(inputs, use_cache=False)
                loss_sum, token_count = token_loss_sum(logits, targets)
            total_loss += float(loss_sum.item())
            total_tokens += token_count
    finally:
        model.train(was_training)
    if total_tokens == 0:
        raise ValueError("验证集没有任何有效目标 token")
    mean_loss = total_loss / total_tokens
    return {"loss": mean_loss,
            "perplexity": math.exp(mean_loss) if mean_loss < 700 else float("inf"),
            "tokens": total_tokens}

# 一次短验证建立训练前基线；结果通常接近 log(vocab_size)，无需强求精确相等。
pretrain_baseline = evaluate_lm(model, pretrain_val_loader, max_batches=4)
print("训练前验证：", pretrain_baseline)

### 7.6 梯度累计与 AMP：一次更新有四个步骤

1. 先取得 `K` 个微批，并数出它们一共含 `N` 个有效目标。
2. 每个微批反向传播 `loss_sum/N`，得到所有有效 token 的平均梯度。
3. 若使用 CUDA fp16，先 `unscale_`，再裁剪梯度，最后尝试更新参数。
4. 清空梯度，为下一次更新开始累计。

AMP 在前向中使用合适的精度；参数和 AdamW 状态仍保持通常的 float32。CUDA bf16 一般不需要缩放器，CPU 路径关闭缩放。**不能先裁剪缩放后的梯度，也不能把不等长微批的均值直接除以 K。**

\[
\nabla\mathcal L=\sum_{k=1}^{K}\frac{N_k}{\sum_j N_j}\nabla\overline{\mathcal L}_k.
\]

In [ ]:
def make_scaler():
    # 只有 fp16 CUDA 路径开启缩放；bf16 和 CPU 仍使用同一调用接口。
    return torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda" and AMP_DTYPE == torch.float16)


def train_one_update(model, micro_batches, optimizer, scaler):
    total_tokens = sum(int((targets != -100).sum().item()) for _, targets in micro_batches)
    if total_tokens == 0:
        raise ValueError("一次更新没有任何有效目标 token")
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    for inputs, targets in micro_batches:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        with amp_context():
            logits, _ = model(inputs, use_cache=False)
            loss_sum, _ = token_loss_sum(logits, targets)
            loss = loss_sum / total_tokens
        if not torch.isfinite(loss).item():
            raise FloatingPointError("损失出现 NaN/Inf；请检查数据、学习率和精度")
        scaler.scale(loss).backward()
        total_loss += float(loss_sum.detach().item())
    # 顺序必须是：还原梯度 → 裁剪 → optimizer step → 更新 scale。
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    if not scaler.is_enabled() and not torch.isfinite(grad_norm).item():
        raise FloatingPointError("未启用 scaler 时梯度出现 NaN/Inf，停止本次更新")
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    successful = scaler.get_scale() >= old_scale
    optimizer.zero_grad(set_to_none=True)
    return {"loss": total_loss / total_tokens, "tokens": total_tokens,
            "grad_norm": float(grad_norm.item()), "successful": successful}

### 7.7 为什么 checkpoint 不只是一个权重文件？

继续训练还需要 AdamW 的一阶、二阶统计量、AMP scale、成功更新数、历史记录与随机状态。还要确认当前模型配置、分词器 SHA256 和数据指纹与保存时一致，避免把同一个 token ID 解释成不同文本。

下面先写临时文件，`flush + fsync` 后用 `os.replace` 替换目标文件。检查点仅包含 Tensor、字符串、数字、列表和字典等安全类型；NumPy 随机状态转为普通数值与 Tensor，因此可以用 `torch.load(..., weights_only=True)` 读取。

**恢复的边界：**保存 RNG 和采样 generator 便于继续训练，但没有保存当前 DataLoader 迭代器中尚未消费的索引。重新创建迭代器后，样本次序可能不同；本教程不承诺逐批精确重放。CUDA 内核也不一定逐位确定。

In [ ]:
import hashlib


def checkpoint_metadata(model):
    model_config = getattr(model, "cfg", getattr(model, "config", cfg))
    # 数据指纹在数据准备章计算，包含三份 bin 与分词器的内容哈希。
    tokenizer_sha = hashlib.sha256((WORK_DIR / "tokenizer.json").read_bytes()).hexdigest()
    summary = {name: {"tokens": int(len(values)), "dtype": str(values.dtype)}
               for name, values in [("train", train_tokens), ("val", val_tokens), ("test", test_tokens)]}
    return {"config": asdict(model_config), "tokenizer_sha256": tokenizer_sha,
            "data_fingerprint": str(DATA_FINGERPRINT), "data_summary": summary}


def capture_rng_state():
    numpy_state = np.random.get_state()
    return {"python": random.getstate(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "numpy": {"algorithm": numpy_state[0],
                      "keys": torch.tensor(numpy_state[1].astype(np.int64)),
                      "position": int(numpy_state[2]), "has_gauss": int(numpy_state[3]),
                      "cached_gaussian": float(numpy_state[4])}}


def restore_rng_state(state):
    random.setstate(state["python"])
    torch.set_rng_state(state["torch"].cpu())
    if state["cuda"] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([value.cpu() for value in state["cuda"]])
    numpy_state = state["numpy"]
    np.random.set_state((numpy_state["algorithm"],
                         numpy_state["keys"].cpu().numpy().astype(np.uint32),
                         numpy_state["position"], numpy_state["has_gauss"],
                         numpy_state["cached_gaussian"]))


def save_checkpoint(path, model, optimizer, scaler, step, history, sampler_generator=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {"format_version": 1, **checkpoint_metadata(model),
               "model": model.state_dict(), "optimizer": optimizer.state_dict(),
               "scaler": scaler.state_dict(), "step": int(step), "history": history,
               "rng": capture_rng_state(),
               "sampler_generator": None if sampler_generator is None else sampler_generator.get_state(),
               "stage_data_summary": getattr(model, "_training_data_summary", None),
               "training_config": getattr(model, "_training_run_config", {})}
    temporary = path.with_name(path.name + f".{os.getpid()}.tmp")
    try:
        with temporary.open("wb") as file:
            torch.save(payload, file)
            file.flush()
            os.fsync(file.fileno())
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()

In [ ]:
def load_checkpoint(path, model, optimizer=None, scaler=None, sampler_generator=None):
    # 保持 weights_only=True，不为不兼容的文件自动退回任意 pickle 执行。
    checkpoint = torch.load(Path(path), map_location="cpu", weights_only=True)
    required = {"format_version", "config", "tokenizer_sha256", "data_fingerprint", "data_summary",
                "model", "optimizer", "scaler", "step", "history", "rng", "sampler_generator"}
    if not required.issubset(checkpoint) or checkpoint["format_version"] != 1:
        raise ValueError("不支持的 checkpoint 格式")
    expected = checkpoint_metadata(model)
    # 先检查兼容性，再修改模型与 optimizer，防止部分加载。
    for key in ["config", "tokenizer_sha256", "data_fingerprint", "data_summary"]:
        if checkpoint[key] != expected[key]:
            raise ValueError(f"checkpoint 与当前 {key} 不一致")
    if hasattr(model, "_training_data_summary"):
        if checkpoint.get("stage_data_summary") != model._training_data_summary:
            raise ValueError("checkpoint 与本阶段训练/验证数据摘要不一致")
    model.load_state_dict(checkpoint["model"], strict=True)
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if scaler is not None:
        scaler.load_state_dict(checkpoint["scaler"])
    restore_rng_state(checkpoint["rng"])
    if sampler_generator is not None and checkpoint["sampler_generator"] is not None:
        sampler_generator.set_state(checkpoint["sampler_generator"].cpu())
    return checkpoint


def loader_generator(loader):
    # RandomSampler 的 generator 控制随机窗口；SFT shuffle 也使用该接口。
    return getattr(loader.sampler, "generator", None) or loader.generator


def describe_loader_data(loader):
    dataset = loader.dataset
    description = {"examples": int(len(dataset))}
    if isinstance(dataset, Subset):
        description["subset_indices"] = [int(index) for index in dataset.indices]
        dataset = dataset.dataset
    if hasattr(dataset, "records_sha256"):
        description.update({"records_sha256": dataset.records_sha256,
                            "dropped": int(dataset.dropped)})
    if hasattr(dataset, "seq_len"):
        description["seq_len"] = int(dataset.seq_len)
    return description

### 7.8 把小函数串成训练循环

`train_lm(..., steps=S)` 的含义是训练到总计 `S` 次成功更新。恢复时从 checkpoint 的 `step` 接着计数；例如文件保存了 `12` 次更新，`steps=20` 还会运行 `8` 次。

循环只做调度：获取微批 → 设置本次学习率 → 调用一次更新 → 定期验证、保存。fp16 梯度溢出时 scaler 会跳过参数更新；这次不增加 `step`，并限制连续跳过次数以免无限循环。历史中的 `tokens` 是该次成功更新处理的有效目标数。

恢复必须保持阶段、峰值学习率和累计设置一致；如果增大 `steps`，未来的 warmup/余弦计划会随总目标重新计算，这不是原计划的逐位延续。

In [ ]:
def train_lm(model, loader, val_loader, steps, lr=3e-4, grad_accum=1,
             stage="pretrain", resume_from=None):
    steps, grad_accum = int(steps), int(grad_accum)
    if steps < 1 or grad_accum < 1 or lr <= 0 or len(loader) == 0:
        raise ValueError("steps、grad_accum、lr 和训练 loader 大小必须为正")
    optimizer, scaler = make_optimizer(model, lr), make_scaler()
    sampler_generator = loader_generator(loader)
    model._training_data_summary = {"train": describe_loader_data(loader),
                                    "validation": describe_loader_data(val_loader)}
    history, step = [], 0
    if resume_from is not None:
        checkpoint = load_checkpoint(resume_from, model, optimizer, scaler, sampler_generator)
        saved_run = checkpoint.get("training_config", {})
        for key, value in [("stage", stage), ("lr", float(lr)), ("grad_accum", grad_accum)]:
            if key in saved_run and saved_run[key] != value:
                raise ValueError(f"恢复训练时 {key} 必须与 checkpoint 一致")
        history, step = list(checkpoint["history"]), int(checkpoint["step"])
        print(f"恢复到 update={step}；新目标为 {steps}，采样顺序不保证精确重放")
    warmup_steps = max(1, int(steps * 0.1))
    model._training_run_config = {"stage": stage, "lr": float(lr), "grad_accum": grad_accum,
                                  "steps": steps, "warmup_steps": warmup_steps}
    eval_every = max(1, int(os.environ.get("LLM_EVAL_EVERY", max(1, steps // 4))))
    eval_max_batches = 8 if PROFILE == "smoke" else 20
    iterator = iter(loader)
    skipped = 0
    total_tokens = sum(int(item["tokens"]) for item in history)
    model.train()
    while step < steps:
        micro_batches = []
        for _ in range(grad_accum):
            try:
                micro_batches.append(next(iterator))
            except StopIteration:
                iterator = iter(loader)
                micro_batches.append(next(iterator))
        current_lr = scheduled_lr(step, steps, lr, warmup_steps)
        for group in optimizer.param_groups:
            group["lr"] = current_lr
        started = time.perf_counter()
        update = train_one_update(model, micro_batches, optimizer, scaler)
        if not update.pop("successful"):
            skipped += 1
            print("AMP 跳过溢出的更新；当前 scale：", scaler.get_scale())
            if skipped >= 20:
                raise FloatingPointError("连续跳过 20 次更新，请降低学习率或改用 bf16")
            continue
        skipped = 0
        step += 1
        total_tokens += update["tokens"]
        record = {"step": step, **update, "lr": current_lr, "total_tokens": total_tokens,
                  "seconds": time.perf_counter() - started}
        if step % eval_every == 0 or step == steps:
            validation = evaluate_lm(model, val_loader, max_batches=eval_max_batches)
            record["val_loss"] = validation["loss"]
            record["val_tokens"] = validation["tokens"]
        history.append(record)
        if "val_loss" in record:
            save_checkpoint(WORK_DIR / f"{stage}_last.pt", model, optimizer, scaler,
                            step, history, sampler_generator)
            print(f"[{stage}] update {step}/{steps}: train={record['loss']:.4f}, "
                  f"val={record['val_loss']:.4f}, lr={current_lr:.2e}")
    return history

### 7.9 现在运行一次短预训练

默认 smoke 和 full 都为 `20` 次短更新，不会自动开始长训练。可以在运行本章前设置 `LLM_PT_STEPS`、`LLM_PT_BATCH_SIZE` 和 `LLM_PT_GRAD_ACCUM`。full 运行需先准备前章要求的本地正式语料。

RTX 4060 Laptop 的 8 GiB 显存建议 full 从 `batch_size=1` 开始，借助前章的梯度检查点和本章的累计控制显存。出现 OOM 时先减小序列长度或微批，并重新建立兼容的配置与数据 loader。**短试训的成功标准是流程完整、损失有限、文件可恢复；不强求每一批损失下降。**

In [ ]:
PT_STEPS = int(os.environ.get("LLM_PT_STEPS", "20"))
PT_RESUME = os.environ.get("LLM_PT_RESUME") or None
pretrain_history = train_lm(
    model, pretrain_loader, pretrain_val_loader, steps=PT_STEPS,
    lr=3e-4, grad_accum=PT_GRAD_ACCUM, stage="pretrain", resume_from=PT_RESUME,
)
assert pretrain_history and pretrain_history[-1]["step"] >= PT_STEPS
assert all(math.isfinite(row["loss"]) for row in pretrain_history)
print("成功更新：", pretrain_history[-1]["step"], "有效 token：", pretrain_history[-1]["total_tokens"])

In [ ]:
def plot_training_history(history, title):
    # 训练曲线是随机小批损失；验证曲线是固定验证子集的加权平均。
    plt.figure(figsize=(7, 3))
    plt.plot([row["step"] for row in history], [row["loss"] for row in history], label="train")
    checked = [row for row in history if "val_loss" in row]
    plt.plot([row["step"] for row in checked], [row["val_loss"] for row in checked],
             "o-", label="validation")
    plt.xlabel("optimizer update")
    plt.ylabel("cross entropy / supervised token")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

plot_training_history(pretrain_history, "Short pretraining")

In [ ]:
# 只检查保存内容，不覆盖当前模型或继续训练。
pretrain_file = WORK_DIR / "pretrain_last.pt"
checkpoint_preview = torch.load(pretrain_file, map_location="cpu", weights_only=True)
assert checkpoint_preview["step"] == pretrain_history[-1]["step"]
assert checkpoint_preview["tokenizer_sha256"] == checkpoint_metadata(model)["tokenizer_sha256"]
print("checkpoint：", pretrain_file.resolve())
print("保存的字段：", sorted(checkpoint_preview))
print("数据摘要：", checkpoint_preview["data_summary"])
del checkpoint_preview  # full 权重很大，检查之后及时释放这份 CPU 副本。

**本章练习与观察**

- 把 toy 窗口 `seq_len` 改成 `2`，手写输入和目标后再执行自检。
- 固定同一批 `[B,T]` 数据，对比一个大批与两个微批的累计梯度；关闭 dropout，并允许浮点误差。SFT 中长度不同的批次更能展示 token 加权的重要性。
- 画出 `warmup_steps=1` 和 `warmup_steps=5` 的学习率曲线，解释峰值位置。
- 在已有 checkpoint 后设置 `LLM_PT_RESUME` 与更大的 `LLM_PT_STEPS`，确认恢复从已保存的成功更新数接续，并理解为什么未保存的迭代游标会改变采样顺序。
- 正式训练要在独立验证集上比较更长训练与多个设置；本章的演示曲线不能支持模型语言能力的结论。

## 8. SFT：教预训练模型按对话格式回答

预训练预测文本流中的所有 token；监督微调（SFT）给出问题和示范回答，主要优化回答位置。参数还是同一组 `model`，损失仍然是 next-token 交叉熵。

这一章先看对话格式和标签掩码，再加载示范数据、短训练、对比留出题。默认算术样本是教学用小数据，20 步不能证明数学推理能力或广泛的指令遵循。正式入口是本地 JSONL 数据文件。

### 8.1 prompt、回答和 EOS 应怎样排在一起？

单轮格式：`[BOS, USER, 问题..., EOS, ASSISTANT, 回答..., EOS]`。

推理 prompt 停在 `ASSISTANT` 标记后；训练标签只保留回答 token 和回答 `EOS`。回答首 token 的目标位于 `ASSISTANT` 的输出位置，所以首 token 也必须受监督。`encode_sft` 返回**尚未移位**的等长 `ids, labels`；移位只在 `SFTDataset` 中做一次。

多轮格式重复用户/助手段；可以在开头加 `SYSTEM`。每一轮 assistant 内容都监督，system、user 和角色标记全部设成 `-100`。

In [ ]:
def encode_prompt(question):
    # 没有 system 消息的单轮 prompt，与下面的训练格式前缀完全一致。
    return [BOS_ID, USER_ID] + tokenizer.encode(str(question)) + [EOS_ID, ASSISTANT_ID]


def encode_sft(messages):
    if not isinstance(messages, list) or not messages:
        raise ValueError("messages 必须是非空列表")
    role_ids = {"system": SYSTEM_ID, "user": USER_ID, "assistant": ASSISTANT_ID}
    ids, labels = [BOS_ID], [-100]
    previous_role = None
    for index, message in enumerate(messages):
        role, content = message.get("role"), message.get("content")
        if role not in role_ids or not isinstance(content, str):
            raise ValueError("每条消息需要合法 role 和字符串 content")
        allowed = ((role == "system" and index == 0) or
                   (role == "user" and previous_role in (None, "system", "assistant")) or
                   (role == "assistant" and previous_role == "user"))
        if not allowed:
            raise ValueError("对话必须为可选 system、user/assistant 交替，并以 assistant 结束")
        content_ids = tokenizer.encode(content)
        segment = [role_ids[role]] + content_ids + [EOS_ID]
        ids.extend(segment)
        # assistant 角色标记本身不监督，回答与终止符都监督。
        labels.extend([-100] + content_ids + [EOS_ID] if role == "assistant" else [-100] * len(segment))
        previous_role = role
    if previous_role != "assistant":
        raise ValueError("SFT 样本必须包含回答并以 assistant 结束")
    return ids, labels

In [ ]:
mask_demo_messages = [{"role": "user", "content": "1 + 2 = ?"},
                      {"role": "assistant", "content": "3"}]
mask_ids, mask_labels = encode_sft(mask_demo_messages)
prompt_ids = encode_prompt(mask_demo_messages[0]["content"])
assert mask_ids[:len(prompt_ids)] == prompt_ids
assert all(value == -100 for value in mask_labels[:len(prompt_ids)])
assert mask_labels[-1] == EOS_ID
assert sum(value != -100 for value in mask_labels) == len(tokenizer.encode("3")) + 1
first_answer_position = len(prompt_ids)
assert mask_ids[first_answer_position - 1] == ASSISTANT_ID
print("目标位置：", [(i, tokenizer.decode([token]), label)
                    for i, (token, label) in enumerate(zip(mask_ids, mask_labels))][-12:])
print("首回答 token 由输入位置", first_answer_position - 1, "预测；EOS 也受监督")

# 多轮自检：两个 assistant 回答的 EOS 都参加损失。
multi_ids, multi_labels = encode_sft(mask_demo_messages + [
    {"role": "user", "content": "2 + 2 = ?"}, {"role": "assistant", "content": "4"}])
assert sum(label == EOS_ID for label in multi_labels) == 2

### 8.2 变长样本如何组成 `[B,T]`？

`SFTDataset` 把每条对话变成 `x=ids[:-1]`、`y=labels[1:]`；collate 在右边补齐 `x` 为 `PAD_ID`、`y` 为 `-100`。因果注意力中，右侧 padding 不会被左侧有效 token 看见，补齐位置也不进入损失。

本教程不默默截断过长样本，以免把回答或 EOS 切掉；它们会被跳过并报告数量。正式数据应先统计长度，必要时按完整轮次切分，或使用合适的更长上下文配置。

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, records, max_seq_len=None):
        self.examples = []
        self.dropped = 0
        records = list(records)
        # 保存规范化内容哈希，恢复 SFT 时检查的就是这份记录，而非只检查预训练 bin。
        self.records_sha256 = hashlib.sha256(
            json.dumps(records, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
        ).hexdigest()
        max_seq_len = cfg.max_seq_len if max_seq_len is None else int(max_seq_len)
        for record in records:
            ids, labels = encode_sft(record["messages"])
            if len(ids) - 1 > max_seq_len:
                self.dropped += 1
                continue
            self.examples.append((torch.tensor(ids[:-1], dtype=torch.long),
                                  torch.tensor(labels[1:], dtype=torch.long)))
        if not self.examples:
            raise ValueError("没有可用 SFT 样本：请检查长度和消息格式")
        print(f"SFTDataset：保留 {len(self.examples)} 条，跳过过长样本 {self.dropped} 条")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def sft_collate(batch):
    length = max(len(inputs) for inputs, _ in batch)
    inputs = torch.full((len(batch), length), PAD_ID, dtype=torch.long)
    targets = torch.full((len(batch), length), -100, dtype=torch.long)
    for row, (item_inputs, item_targets) in enumerate(batch):
        inputs[row, :len(item_inputs)] = item_inputs
        targets[row, :len(item_targets)] = item_targets
    return inputs, targets

# 可观察自检：仅短样本的尾部被补齐，且补齐目标不参加损失。
collate_demo = SFTDataset([{"messages": mask_demo_messages}, {"messages": [
    {"role": "user", "content": "1 + 2 = ? 请写整数"}, {"role": "assistant", "content": "3"}]}])
demo_x, demo_y = sft_collate([collate_demo[0], collate_demo[1]])
first_length = len(collate_demo[0][0])
assert torch.all(demo_x[0, first_length:] == PAD_ID)
assert torch.all(demo_y[0, first_length:] == -100)
assert demo_y[0, first_answer_position - 1] == mask_ids[first_answer_position]
print("补齐后形状：", tuple(demo_x.shape), "每行有效目标：", (demo_y != -100).sum(dim=1).tolist())

In [ ]:
def load_sft_jsonl(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
                if not isinstance(record, dict) or "messages" not in record:
                    raise ValueError("需要含 messages 的 JSON 对象")
                encode_sft(record["messages"])  # 提前验证格式并给出准确行号。
            except (ValueError, TypeError, AttributeError, KeyError) as error:
                raise ValueError(f"{path} 第 {line_number} 行错误：{error}") from error
            records.append(record)
    if not records:
        raise ValueError(f"{path} 不含任何有效记录")
    return records


def make_sft_loaders(train_records, val_records, batch_size=None):
    batch_size = (2 if PROFILE == "smoke" else 1) if batch_size is None else int(batch_size)
    train_dataset = SFTDataset(train_records)
    val_dataset = SFTDataset(val_records)
    generator = torch.Generator().manual_seed(2027)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, generator=generator,
        collate_fn=sft_collate, num_workers=0, pin_memory=DEVICE.type == "cuda",
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, collate_fn=sft_collate,
        num_workers=0, pin_memory=DEVICE.type == "cuda",
    )
    return train_loader, val_loader

### 8.3 教学数据：SFT 训练题与本阶段留出题分开

遍历 `a,b ∈ {0,...,9}` 可以得到 100 道加法题。预先选出 6 对作为 SFT 阶段的留出题，其他 94 对用于教学训练；验证损失和生成都在这些题上观察。这里的划分只保证它们不在 SFT 训练记录中；是否曾出现在预训练语料里，要单独审查前章的数据。它们共享相同模板，因此仍然是很窄的检验，不能据此推断对新任务的泛化。

做纯粹的记忆能力练习时，可以把训练集缩到 1～4 条并重复训练更多次，观察是否能拟合这些示范；这只能证明能拟合训练例。

In [ ]:
def arithmetic_question(a, b):
    return f"请只给出整数结果：{a} + {b} 等于多少？"


def arithmetic_record(a, b):
    return {"messages": [{"role": "user", "content": arithmetic_question(a, b)},
                         {"role": "assistant", "content": str(a + b)}]}

heldout_pairs = {(2, 7), (4, 6), (8, 3), (9, 9), (0, 5), (6, 1)}
demo_sft_records = [arithmetic_record(a, b) for a in range(10) for b in range(10)
                    if (a, b) not in heldout_pairs]
demo_sft_val_records = [arithmetic_record(a, b) for a, b in sorted(heldout_pairs)]
assert len(demo_sft_records) == 94 and len(demo_sft_val_records) == 6
train_questions = {record["messages"][0]["content"] for record in demo_sft_records}
val_questions = {record["messages"][0]["content"] for record in demo_sft_val_records}
assert train_questions.isdisjoint(val_questions)
print("教学训练题：", len(train_questions), "留出题：", sorted(val_questions))

### 8.4 正式本地 JSONL 入口

设置 `LLM_SFT_TRAIN_JSONL` 和 `LLM_SFT_VAL_JSONL`，两个文件必须分别准备；每行一个记录。例如：

也可以直接把两个文件放在默认的 `data/sft_train.jsonl` 和 `data/sft_val.jsonl`；代码会自动检测。环境变量可指定其他路径，且优先于默认位置。

```json
{"messages":[{"role":"user","content":"请解释梯度累计"},{"role":"assistant","content":"多个小批先累计梯度，再执行一次参数更新。"}]}
```

支持开头的 system 与多轮 user/assistant 对话；训练记录应以 assistant 结束。正式数据需要在去重、来源与权限确认后划分，避免同一对话或近重复同时进入训练与验证。只提供一个文件时会报错，不从训练记录中临时抽几条充当独立验证。

为保持无下载的 smoke 路径，环境变量与默认本地文件均未提供时使用上述算术教学集。full 模式也可以执行这个机制演示；它不会把算术模板变成正式 SFT 语料。

In [ ]:
sft_train_path = os.environ.get("LLM_SFT_TRAIN_JSONL") or ("data/sft_train.jsonl" if Path("data/sft_train.jsonl").exists() else None)
sft_val_path = os.environ.get("LLM_SFT_VAL_JSONL") or ("data/sft_val.jsonl" if Path("data/sft_val.jsonl").exists() else None)
if bool(sft_train_path) != bool(sft_val_path):
    raise ValueError("请同时设置 LLM_SFT_TRAIN_JSONL 和 LLM_SFT_VAL_JSONL")
if sft_train_path:
    sft_train_records = load_sft_jsonl(sft_train_path)
    sft_val_records = load_sft_jsonl(sft_val_path)
    print("SFT 来源：本地 JSONL")
else:
    sft_train_records, sft_val_records = demo_sft_records, demo_sft_val_records
    print("SFT 来源：算术教学示例")
sft_train_loader, sft_val_loader = make_sft_loaders(sft_train_records, sft_val_records)
sft_x, sft_y = next(iter(sft_train_loader))
assert sft_x.shape == sft_y.shape and torch.any(sft_y != -100)
print("SFT 训练小批：", tuple(sft_x.shape), "有效回答 token：", int((sft_y != -100).sum()))

### 8.5 先记录微调前的生成，再训练

训练损失回答的是“示范答案的概率是否提高”；生成还会受解码方法影响。这里固定使用贪心解码（`temperature=0`），并截取 prompt 之后、首个 EOS 之前的内容。所有留出题保持同样的推理格式。

在本地正式 JSONL 模式下，只用验证损失比较自己的数据；下列算术输出始终只是教学用 probe，不属于任意正式数据集的评估。

In [ ]:
def answer_question(question, max_new_tokens=12):
    prompt = encode_prompt(question)
    generated = generate(model, prompt, max_new_tokens=max_new_tokens,
                         temperature=0.0, use_cache=True)
    answer_ids = generated[len(prompt):]
    if EOS_ID in answer_ids:
        answer_ids = answer_ids[:answer_ids.index(EOS_ID)]
    return tokenizer.decode(answer_ids).strip()

sft_probe_pairs = sorted(heldout_pairs)[:3]
sft_before_answers = {pair: answer_question(arithmetic_question(*pair)) for pair in sft_probe_pairs}
sft_before_validation = evaluate_lm(model, sft_val_loader, max_batches=8 if PROFILE == "smoke" else 20)
print("SFT 前验证：", sft_before_validation)
for pair, answer in sft_before_answers.items():
    print(arithmetic_question(*pair), "→", repr(answer), "期望：", sum(pair))

### 8.6 复用同一训练循环，只换数据和峰值学习率

SFT 的有效 token 仅来自回答，短答案尤其需要正确累计权重。本节建立新的 AdamW 状态，因为我们开始新的训练阶段；预训练模型参数继续作为起点。`LLM_SFT_RESUME` 用于恢复同一 SFT 阶段的检查点。

默认 smoke 和 full 都做 `20` 次短更新，可用 `LLM_SFT_STEPS` 改变。正式微调通常需另行设定数据量、轮次与验证频率，较低学习率只是这里的起点。

In [ ]:
SFT_STEPS = int(os.environ.get("LLM_SFT_STEPS", "20"))
SFT_GRAD_ACCUM = int(os.environ.get("LLM_SFT_GRAD_ACCUM", "2" if PROFILE == "smoke" else "16"))
sft_history = train_lm(
    model, sft_train_loader, sft_val_loader, steps=SFT_STEPS,
    lr=2e-4, grad_accum=SFT_GRAD_ACCUM, stage="sft",
    resume_from=os.environ.get("LLM_SFT_RESUME") or None,
)
assert sft_history and sft_history[-1]["step"] >= SFT_STEPS
assert all(math.isfinite(row["loss"]) for row in sft_history)
plot_training_history(sft_history, "Short SFT: answer-only loss")

In [ ]:
sft_after_validation = evaluate_lm(model, sft_val_loader, max_batches=8 if PROFILE == "smoke" else 20)
sft_after_answers = {pair: answer_question(arithmetic_question(*pair)) for pair in sft_probe_pairs}
print("SFT 验证损失：", sft_before_validation["loss"], "→", sft_after_validation["loss"])
for pair in sft_probe_pairs:
    print(f"{pair[0]}+{pair[1]}：前={sft_before_answers[pair]!r}，"
          f"后={sft_after_answers[pair]!r}，期望={sum(pair)}")
if not sft_train_path:
    # 严格字符串比较，不用解析器修补输出后再声称回答正确。
    heldout_answers = {pair: answer_question(arithmetic_question(*pair)) for pair in sorted(heldout_pairs)}
    exact_matches = sum(answer == str(sum(pair)) for pair, answer in heldout_answers.items())
    print(f"教学留出题严格匹配：{exact_matches}/{len(heldout_answers)}；样本太少，不代表广泛能力")
print("SFT checkpoint：", (WORK_DIR / "sft_last.pt").resolve())

**本章练习与下一步**

- 把一条对话写成两轮，逐 token 检查哪些标签是 `-100`，解释两个回答 EOS 为什么都受监督。
- 故意把 padding 目标设成 `PAD_ID`，观察损失计算目标数的变化，再改回 `-100`；这展示了掩码如何改变训练任务。
- 只用 4 条示范训练较多步，记录训练例拟合和留出题结果。训练例拟合成功不等于获得通用加法能力。
- 接入自己的本地训练/验证 JSONL，报告保留与跳过数量、验证有效 token 数及曲线；不要只展示几个挑选出的正确回答。

此时 `model` 保留 SFT 后的参数，`sft_history` 保留更新记录，后续章节可以继续用它作为强化学习的初始策略。正式实验应保存 SFT 检查点作为可复用起点，并按下一章的规则另建参考模型。

## 9. RLVR：从“模仿答案”到“提高高奖励答案的概率”

SFT 的老师给出一条正确回答，交叉熵直接提高它的概率。强化学习先让模型自己抽样，再用奖励评价回答；它并不自动提供正确答案。

| 方法 | 反馈从哪里来 | 本教程是否实现 |
|---|---|---|
| RLHF | 人类偏好通常训练奖励模型，再用强化学习优化策略 | 只解释，不训练奖励模型 |
| RLVR | 程序能验证答案，例如算术答案或代码测试 | 实现严格整数算术奖励 |
| DPO | 对“选中/拒绝”的回答对直接优化偏好目标 | 只解释，不把它当作在线 RL |
| GRPO | 同一问题抽样一组回答，用组内相对奖励替代价值模型 | 实现一次更新的教学简化版 |

“RLVR”描述奖励来源，“GRPO”描述优化方法，二者可以组合。我们先学 REINFORCE 的直觉，再加入概率比、裁剪和参考策略约束。GRPO 的原始方法见 [DeepSeekMath](https://arxiv.org/abs/2402.03300)。下面省略完整生产系统的分布式采样、数据筛选、多次 PPO 更新和复杂奖励组合。

**学习目标：** 能解释“奖励如何改变下一 token 的概率”；能发现没有奖励差异时为何不该声称训练成功。


### 9.1 奖励先写成一个可以单独测试的程序

对题目 `2 + 3 = ?`，回答 `5` 获得 1，其他回答获得 0。本节只接受纯整数，允许首尾空白与正负号；`先猜4再猜5`、`答案是5` 都得 0。这是一个刻意简单的**格式与正确性联合奖励**，不是推理质量评价。

不要用 `eval(answer)` 执行模型文本，也不要从多个数字中挑出碰巧正确的那个。验证器本身就是训练目标的一部分，漏洞会变成可利用的捷径。

预计输出：`[1.0, 1.0, 0.0, 0.0, 0.0]`。自检：空回答必须为 0，含多个候选数字的回答必须为 0。


In [ ]:
def arithmetic_reward(answer: str, target: int) -> float:
    # fullmatch 要求整个回答符合一个整数，而非在长文本中找到某个整数。
    match = re.fullmatch(r"\s*([+-]?[0-9]+)\s*", answer)
    if match is None:
        return 0.0
    try:
        value = int(match.group(1))
    except ValueError:
        return 0.0
    return float(value == int(target))


reward_examples = ["5", "  +5\n", "4", "错误0但最后正确5", ""]
print([arithmetic_reward(text, 5) for text in reward_examples])
assert arithmetic_reward("-3", -3) == 1.0
assert arithmetic_reward("5或6", 5) == 0.0
assert arithmetic_reward("答案是5", 5) == 0.0
assert arithmetic_reward("", 0) == 0.0


### 9.2 相对奖励只在同一道题内部比较

同一问题抽样 $G=4$ 个回答，奖励记作 $r_1,\ldots,r_G$。组内优势为

$$A_i=\frac{r_i-\operatorname{mean}(r)}{\operatorname{std}(r)},\qquad
\operatorname{std}(r)=\sqrt{\frac1G\sum_i(r_i-\operatorname{mean}(r))^2}.$$

这里标准差使用总体定义，即 PyTorch 的 `unbiased=False`。`[0, 1, 0, 1]` 的优势是 `[-1, 1, -1, 1]`，不是把不同题目的难度混成一个基线。`[0,0,0,0]` 与 `[1,1,1,1]` 都没有组内相对信号：优势设为 0，并跳过策略更新。

Shape：单组输入 `[G]` → `[G]`；多个问题输入 `[B,G]` → `[B,G]`，每一行独立计算。预计每行优势均值接近 0。练习：预测 `[0,0,0,1]` 的四个优势符号。


In [ ]:
def group_advantages(rewards: torch.Tensor) -> torch.Tensor:
    # 末维是一道题的回答组，前面的维度不能参与这道题的归一化。
    values = rewards.detach().float()
    if values.ndim == 0 or values.shape[-1] == 0:
        raise ValueError("奖励至少需要一个非空组维度")
    centered = values - values.mean(dim=-1, keepdim=True)
    std = values.std(dim=-1, unbiased=False, keepdim=True)
    safe_std = std.clamp_min(1e-8)
    return torch.where(std > 1e-8, centered / safe_std, torch.zeros_like(centered))


reward_groups = torch.tensor([[0., 1., 0., 1.], [0., 0., 0., 0.], [1., 1., 1., 1.]])
advantage_groups = group_advantages(reward_groups)
print("每一行是一道题：\n", advantage_groups)
assert torch.allclose(advantage_groups.mean(dim=-1), torch.zeros(3), atol=1e-6)
assert torch.equal(advantage_groups[1:], torch.zeros(2, 4))


### 9.3 一个一定看得见梯度方向的小实验

语言模型可能暂时一个正确整数都采不到。先用只有两个候选的概率模型观察机制：候选 0 是错误答案，候选 1 是正确答案。两个可训练 logits 经 softmax 得到概率 $p$。

REINFORCE 的样本损失是 $-A\log p(\text{抽到的答案})$。下格人为指定一次“正确答案被抽到”的事件，优势设为 +1，只为了隔离梯度机制。**这不是 TinyLLM 的 rollout，不能把它计入模型正确率或 RL 成果。**

预计：更新前正确答案概率约 0.5，更新后大于 0.5。练习：把优势改为 −1，概率应朝哪个方向移动？为什么奖励/优势必须视作常量？


In [ ]:
# 只有两个参数的概率实验，完全不改变已经训练的 model。
toy_logits = nn.Parameter(torch.zeros(2))
toy_optimizer = torch.optim.SGD([toy_logits], lr=0.5)
before = toy_logits.softmax(dim=0).detach().clone()
toy_advantage = torch.tensor(1.0)
toy_loss = -toy_advantage.detach() * toy_logits.log_softmax(dim=0)[1]
toy_optimizer.zero_grad()
toy_loss.backward()
print("logits 的梯度：", toy_logits.grad.tolist())
toy_optimizer.step()
after = toy_logits.softmax(dim=0).detach()
print("更新前概率：", before.tolist())
print("更新后概率：", after.tolist())
assert after[1] > before[1]
del toy_optimizer, toy_logits


### 9.4 哪些 log probability 属于回答？

设 prompt 有 $P$ 个 token，回答有 $T$ 个 token。拼接输入 shape 为 `[1,P+T]`，模型输出 `[1,P+T,V]`。第 `P-1` 个位置的 logits 预测第一个回答 token；选取位置区间 `[P-1, P+T-1)` 后 shape 为 `[T,V]`，再按实际回答 token `gather` 得到 `[T]`。

回答 EOS 也是模型采取的动作，**其 log probability 参与损失**；prompt、padding 不参与。我们直接使用未 padding 的单条序列，因此回答掩码等价于这个切片。EOS 后不再追加任何 token。

下方函数保留梯度；只有旧策略与参考策略的调用放在 `no_grad()` 中。完整词表 `log_softmax` 同时是概率归一化检查：如果 rollout 先做 top-k 截断，却在这里使用完整分布，训练就悄悄换了策略。


In [ ]:
def completion_log_probs(policy, prompt_ids, completion_ids) -> torch.Tensor:
    if len(prompt_ids) < 1:
        raise ValueError("prompt 至少含一个 token")
    if len(completion_ids) < 1:
        raise ValueError("回答至少含一个 token；EOS 也算回答 token")
    sequence = list(prompt_ids) + list(completion_ids)
    if len(sequence) > policy.cfg.max_seq_len:
        raise ValueError("prompt 与回答的总长度超过模型上下文")
    policy_device = next(policy.parameters()).device
    ids = torch.tensor([sequence], dtype=torch.long, device=policy_device)
    with amp_context():
        logits, _ = policy(ids, use_cache=False)
    p = len(prompt_ids)
    t = len(completion_ids)
    # logits 的位置落后目标 token 一位；float32 归一化更稳定。
    response_logits = logits[0, p - 1:p + t - 1, :].float()
    targets = torch.tensor(completion_ids, dtype=torch.long, device=policy_device)
    selected = F.log_softmax(response_logits, dim=-1).gather(1, targets[:, None]).squeeze(1)
    assert selected.shape == (t,)
    return selected


alignment_prompt = encode_prompt("2+3=?")
alignment_completion = tokenizer.encode("5") + [EOS_ID]
alignment_logp = completion_log_probs(model, alignment_prompt, alignment_completion)
print("prompt 长度：", len(alignment_prompt), "回答长度（含EOS）：", len(alignment_completion))
print("回答 log probability shape：", tuple(alignment_logp.shape))
assert alignment_logp.requires_grad
assert torch.isfinite(alignment_logp).all()
del alignment_logp


### 9.5 从 REINFORCE 到带裁剪的策略目标

对于回答中的一个 token，令

$$\rho_t=\exp(\log\pi_\theta(y_t|x,y_{<t})-\operatorname{stopgrad}(\log\pi_{\rm old}(y_t|x,y_{<t}))),$$
$$k_t=\exp(\log\pi_{\rm ref}-\log\pi_\theta)-(\log\pi_{\rm ref}-\log\pi_\theta)-1,$$
$$L=-\operatorname{mean}_t\big[\min(\rho_t A,\operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)A)-\beta k_t\big].$$

旧策略是在抽样时固定下来的概率，参考策略是 RL 开始前的冻结 SFT 模型。这里的 $k_t$ 是一个常见的非负 KL 估计/惩罚项，并非逐个 token 枚举整个词表的精确 KL。若 token 确实从当前策略抽样，它的期望对应 $\mathrm{KL}(\pi_\theta\|\pi_{\rm ref})$；若多次更新后复用旧 rollout，这种解释通常只是近似。不要把一次抽样值直接当成总体 KL。

每条回答先除以自己的 token 数，再平均回答，避免仅因为回答更长而权重更大。教学实现每组只做**一次**优化器更新：第一次前向时 ratio 应约为 1，裁剪机制的作用通过下一格的人工张量自检展示。它是 GRPO 风格的简化策略梯度，不是完整生产版 GRPO。


In [ ]:
def grpo_objective(new_logp, old_logp, ref_logp, advantage, clip_eps=0.2, beta=0.02):
    if new_logp.shape != old_logp.shape or new_logp.shape != ref_logp.shape:
        raise ValueError("新、旧、参考 log probability 的 shape 必须相同")
    if new_logp.ndim not in (1, 2) or new_logp.shape[-1] == 0:
        raise ValueError("输入应为 [T] 或未 padding 的 [B,T]")
    # 旧策略和参考策略不参与反向传播；奖励优势也不参与求导。
    current = new_logp.float()
    old = old_logp.detach().float()
    reference = ref_logp.detach().float()
    adv = torch.as_tensor(advantage, dtype=current.dtype, device=current.device).detach()
    while adv.ndim < current.ndim:
        adv = adv.unsqueeze(-1)
    ratio = torch.exp(current - old)
    surrogate = torch.minimum(ratio * adv, ratio.clamp(1 - clip_eps, 1 + clip_eps) * adv)
    ref_minus_new = reference - current
    sampled_kl = torch.exp(ref_minus_new) - ref_minus_new - 1.0
    # 末维先按回答长度归一化，再对组内回答平均。
    return (-surrogate + beta * sampled_kl).mean(dim=-1).mean()


# 故意制造 ratio=2：正优势的裁剪目标最多到 1.2。
test_new = torch.tensor([math.log(0.4)], requires_grad=True)
test_old = torch.tensor([math.log(0.2)], requires_grad=True)
test_ref = torch.tensor([math.log(0.4)], requires_grad=True)
test_loss = grpo_objective(test_new, test_old, test_ref, 1.0, beta=0.0)
test_loss.backward()
print("裁剪后的损失（预计 -1.2）：", float(test_loss.detach()))
assert torch.allclose(test_loss.detach(), torch.tensor(-1.2), atol=1e-6)
assert test_old.grad is None and test_ref.grad is None
assert torch.allclose(test_new.grad, torch.zeros_like(test_new))


### 9.6 真实 rollout：抽四次，按回答逐条反向传播

下方 rollout 固定温度为 1，不做 top-k、top-p 或特殊 token 屏蔽。它与 `completion_log_probs` 的完整 softmax 是同一个策略。出现 PAD/BOS/角色 token 的回答奖励为 0，避免把解码器隐藏的特殊 token 当成合法整数。结束 EOS 保留在概率序列中，解码用于奖励时再移除。

抽样和评分都在 `eval()` 模式下完成，更新前向也保持同模式；`eval()` 不会关闭梯度，`no_grad()` 才会。这让含 dropout 的扩展模型也不会在采样/评分间换分布。每条回答立即 `backward()`，不堆四条计算图；旧 log probability 只保存张量，不额外复制一份旧模型。复用前章的 GradScaler 接口：fp16 缩放梯度，bf16/CPU 禁用缩放；先 unscale，再裁剪，最后 step/update，溢出时明确记录跳过。

一组奖励零方差时返回 `skipped=True`，不编造正确回答来强行更新。参考 SFT 模型需要一份冻结副本；8 GB 显存的 full 模式仍应从短上下文、单条微批开始。代码只保留这一份参考模型，RL 结束即释放。


In [ ]:
@torch.no_grad()
def _rl_rollout(policy, prompt_ids, max_new_tokens=8):
    # 使用完整词表概率抽样，故意不应用聊天时常见的截断策略。
    if max_new_tokens < 1:
        raise ValueError("max_new_tokens 必须至少为1")
    limit = min(int(max_new_tokens), policy.cfg.max_seq_len - len(prompt_ids))
    if limit < 1:
        raise ValueError("prompt 没有为回答留下上下文空间")
    policy_device = next(policy.parameters()).device
    sequence = list(prompt_ids)
    completion = []
    for _ in range(limit):
        ids = torch.tensor([sequence], dtype=torch.long, device=policy_device)
        with amp_context():
            logits, _ = policy(ids, use_cache=False)
        probabilities = F.softmax(logits[0, -1].float(), dim=-1)
        token = int(torch.multinomial(probabilities, num_samples=1).item())
        completion.append(token)
        sequence.append(token)
        if token == EOS_ID:
            break
    return completion


def _arithmetic_completion(completion_ids, target):
    # 角色或padding不能被decode隐藏后伪装成有效答案。
    forbidden = {PAD_ID, BOS_ID, USER_ID, ASSISTANT_ID, SYSTEM_ID}
    invalid_special = any(token in forbidden for token in completion_ids)
    answer = tokenizer.decode([token for token in completion_ids if token != EOS_ID]).strip()
    reward = 0.0 if invalid_special else arithmetic_reward(answer, target)
    return answer, reward, invalid_special


def rlvr_step(policy, reference, question, target, optimizer, group_size=4, max_new_tokens=8, scaler=None):
    if group_size < 2:
        raise ValueError("相对奖励至少需要两个回答")
    prompt_ids = encode_prompt(question)
    was_training = policy.training
    policy.eval()
    reference.eval()
    optimizer.zero_grad(set_to_none=True)
    try:
        completions = [_rl_rollout(policy, prompt_ids, max_new_tokens) for _ in range(group_size)]
        scored = [_arithmetic_completion(ids, target) for ids in completions]
        policy_device = next(policy.parameters()).device
        rewards = torch.tensor([entry[1] for entry in scored], device=policy_device)
        advantages = group_advantages(rewards)
        statistics = {
            "question": question, "target": int(target),
            "answers": [entry[0] for entry in scored], "rewards": rewards.cpu().tolist(),
            "advantages": advantages.cpu().tolist(),
            "mean_reward": float(rewards.mean().item()),
            "response_tokens": [len(ids) for ids in completions],
            "empty_answers": sum(entry[0] == "" for entry in scored),
            "invalid_special_answers": sum(entry[2] for entry in scored),
            "skipped": False, "skip_reason": None, "loss": None,
        }
        if rewards.std(unbiased=False).item() <= 1e-8:
            statistics.update(skipped=True, skip_reason="组内奖励零方差，没有相对学习信号")
            return statistics
        scaler = make_scaler() if scaler is None else scaler
        # 在任何参数更新前缓存抽样策略概率，detach 保证 old 固定。
        with torch.no_grad():
            old_logps = [completion_log_probs(policy, prompt_ids, ids).detach() for ids in completions]
            ref_logps = [completion_log_probs(reference, prompt_ids, ids).detach() for ids in completions]
        loss_total = 0.0
        for ids, old_logp, ref_logp, advantage in zip(completions, old_logps, ref_logps, advantages):
            new_logp = completion_log_probs(policy, prompt_ids, ids)
            # 每条微批的权重是1/G；本条的计算图在backward后释放。
            loss = grpo_objective(new_logp, old_logp, ref_logp, advantage) / group_size
            if not torch.isfinite(loss).item():
                raise FloatingPointError("RL损失出现NaN/Inf；本次更新停止，请检查精度和概率数值")
            scaler.scale(loss).backward()
            loss_total += float(loss.detach().item())
            del new_logp, loss
        # 与前章保持同样顺序：unscale → clip → step → update。
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        if not scaler.is_enabled() and not torch.isfinite(grad_norm).item():
            raise FloatingPointError("RL梯度出现NaN/Inf；未启用缩放器，本次更新停止")
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        successful = scaler.get_scale() >= previous_scale and torch.isfinite(grad_norm).item()
        statistics.update(loss=loss_total, grad_norm=float(grad_norm.item()),
                          skipped=not successful,
                          skip_reason=None if successful else "AMP检测到非有限梯度，跳过参数更新")
        return statistics
    finally:
        optimizer.zero_grad(set_to_none=True)
        policy.train(was_training)


### 9.7 两次尝试只验证训练链路

本格默认 smoke 与 full **都只尝试 2 次 RL 更新**。从随机初始化、很少的预训练/SFT 出发，四次采样全得 0 很正常。日志若显示 `跳过`，含义就是“这次没有更新”，不是暗中注入标准答案后的成功。若你要研究能力提升，需自行增加高质量训练数据、SFT 收敛程度、抽样数和训练步数，并另建固定验证集。

保存两个明确标记的检查点：`sft_before_rl` 与 `rlvr_after_attempts`。第二个记录实际更新数，可能为 0。不能仅凭文件名、一次随机正确回答或训练 loss 下降断言 RL 有效。

显存提示：0.1B 的 float32 权重约 0.4 GB；policy + reference 约 0.8 GB，Adam 两个状态约 0.8 GB，加上梯度约 0.4 GB，再加激活与临时张量。实际峰值取决于上下文与实现；此处释放前面阶段的优化器状态后再开始 RL。


In [ ]:
import copy


def save_stage_checkpoint(stage, policy, metadata=None):
    checkpoint_dir = WORK_DIR / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / f"{PROFILE}_{stage}.pt"
    torch.save({
        "stage": stage, "profile": PROFILE, "config": asdict(policy.cfg),
        "state_dict": {name: value.detach().cpu() for name, value in policy.state_dict().items()},
        "metadata": metadata or {},
    }, checkpoint_path)
    return checkpoint_path


SFT_BEFORE_RL_PATH = save_stage_checkpoint("sft_before_rl", model, {"rl_updates": 0})
# 前面的训练优化器是函数内局部变量；重复执行本格时只释放本节自身对象。
for _name in ("rl_optimizer", "rl_scaler", "reference_model"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
reference_model = copy.deepcopy(model).eval().requires_grad_(False)
rl_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6)
rl_scaler = make_scaler()
RL_STEPS = 2 if PROFILE == "smoke" else 2
RL_TRAIN_CASES = [
    (f"请只给出整数结果：{a} + {b} 等于多少？", a + b)
    for a, b in [(2, 3), (5, 4), (7, 2), (1, 8)]
]
print("SFT 检查点：", SFT_BEFORE_RL_PATH)
print("计划 RL 尝试：", RL_STEPS, "，每题采样4条；并不保证产生更新")


In [ ]:
rl_history = []
for rl_index in range(RL_STEPS):
    question, target = RL_TRAIN_CASES[rl_index % len(RL_TRAIN_CASES)]
    stats = rlvr_step(model, reference_model, question, target, rl_optimizer,
                      group_size=4, max_new_tokens=8, scaler=rl_scaler)
    rl_history.append(stats)
    print(f"尝试 {rl_index + 1}/{RL_STEPS}：reward={stats['rewards']}，"
          f"{'跳过' if stats['skipped'] else '已更新'}，loss={stats['loss']}")
    if stats["skipped"]:
        print("跳过原因：", stats["skip_reason"])
    print("真实模型回答：", [repr(answer) for answer in stats["answers"]])
rl_updates = sum(not row["skipped"] for row in rl_history)
RL_AFTER_PATH = save_stage_checkpoint("rlvr_after_attempts", model, {
    "attempts": RL_STEPS, "rl_updates": rl_updates, "group_size": 4,
    "history": rl_history, "note": "教学短训练；不代表能力提升",
})
print(f"实际更新数：{rl_updates}/{RL_STEPS}；检查点：{RL_AFTER_PATH}")
# 后面的评估与聊天只需一个模型，不长期占用参考模型与优化器状态。
del reference_model, rl_optimizer, rl_scaler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 10. 评估：把一次输出与稳定能力分开

这里验证**本教程合成训练集未使用的算术问题**。默认教学数据的训练操作数是 `0..9`；下面只使用 `31..79` 的操作数，RL 训练题也未包含它们。full 的外部语料未必满足这个约定，因此下格扫描实际预训练/SFT train 文本中的算式，排除粗检发现的重叠并显示排除数。真实语料需要独立、完整去污染：这个表达式粗检不保证模型未见过题目的改写、答案或等价形式。验证答案不能追加回训练集后继续称为 heldout。

指标至少同时报告：`correct / total`、空回答数量、含非法特殊 token 的数量。两组随机种子各抽样一次，用来观察采样变化；这是每个种子的单样本正确率，不是 best-of-N、多数投票或 pass@k。smoke 的零分是正常且有意义的结果，说明跑通链路与训练出能力是不同目标。

自检：精确训练问题交集应为空；粗检排除后必须按实际剩余题数统计，全部排除时不报告正确率。


In [ ]:
HELDOUT_PAIRS = [(31, 47), (52, 36), (64, 35), (79, 32), (43, 58), (68, 37),
                 (35, 62), (74, 39), (56, 38), (71, 46), (49, 63), (67, 54)]
heldout_count = 4 if PROFILE == "smoke" else len(HELDOUT_PAIRS)
HELDOUT_CASES = [(f"请只给出整数结果：{a} + {b} 等于多少？", a + b)
                 for a, b in HELDOUT_PAIRS[:heldout_count]]
synthetic_training_questions = set(question for question, _ in RL_TRAIN_CASES)
synthetic_training_questions.update(message["content"] for record in demo_sft_records
                                    for message in record["messages"] if message["role"] == "user")
assert not synthetic_training_questions.intersection(question for question, _ in HELDOUT_CASES)
# 检查前文实际读入的SFT训练记录，而非只检查本节的RL题目。
training_questions = {message["content"] for record in sft_train_records
                      for message in record["messages"] if message["role"] == "user"}
exact_question_overlap = training_questions.intersection(question for question, _ in HELDOUT_CASES)
# 同时检查预训练原文中a+b和b+a两种写法；逐篇流式扫描，不堆全部文本。
heldout_patterns = [(a, b, re.compile(rf"(?<![0-9])(?:{a}\s*\+\s*{b}|{b}\s*\+\s*{a})\s*(?:=|等于)"))
                    for a, b in HELDOUT_PAIRS[:heldout_count]]
heldout_contamination = set()
for a, b in HELDOUT_PAIRS[:heldout_count]:
    if f"请只给出整数结果：{a} + {b} 等于多少？" in exact_question_overlap:
        heldout_contamination.add((a, b))
for training_text in iter_split_text(split_paths["train"]):
    for a, b, pattern in heldout_patterns:
        if pattern.search(training_text):
            heldout_contamination.add((a, b))
for record in sft_train_records:
    for message in record["messages"]:
        for a, b, pattern in heldout_patterns:
            if pattern.search(message["content"]):
                heldout_contamination.add((a, b))
HELDOUT_CASES = [(f"请只给出整数结果：{a} + {b} 等于多少？", a + b)
                 for a, b in HELDOUT_PAIRS[:heldout_count] if (a, b) not in heldout_contamination]
print("表达式粗检排除题数：", len(heldout_contamination), "；剩余验证题数：", len(HELDOUT_CASES))


@torch.no_grad()
def evaluate_arithmetic(policy, cases, seeds=(2027, 2028), max_new_tokens=8):
    result_rows = []
    policy_device = next(policy.parameters()).device
    cuda_devices = ([policy_device.index if policy_device.index is not None else torch.cuda.current_device()]
                    if policy_device.type == "cuda" else [])
    was_training = policy.training
    policy.eval()
    try:
        for seed in seeds:
            records = []
            # 不让评估的种子改变Notebook后续的随机数状态。
            with torch.random.fork_rng(devices=cuda_devices):
                # CPU参考评估只设置CPU种子，避免顺便改变其他GPU的随机数状态。
                torch.random.default_generator.manual_seed(seed)
                if policy_device.type == "cuda":
                    with torch.cuda.device(policy_device):
                        torch.cuda.manual_seed(seed)
                for question, target in cases:
                    completion = _rl_rollout(policy, encode_prompt(question), max_new_tokens)
                    answer, reward, invalid = _arithmetic_completion(completion, target)
                    records.append({"question": question, "target": target, "answer": answer,
                                    "correct": int(reward), "empty": int(answer == ""),
                                    "invalid_special": int(invalid)})
            total = len(records)
            correct = sum(row["correct"] for row in records)
            result_rows.append({"seed": seed, "correct": correct, "total": total,
                                "accuracy": correct / total if total else None,
                                "empty": sum(row["empty"] for row in records),
                                "invalid_special": sum(row["invalid_special"] for row in records),
                                "records": records})
    finally:
        policy.train(was_training)
    return result_rows


heldout_results = evaluate_arithmetic(model, HELDOUT_CASES)
for row in heldout_results:
    if row["total"] == 0:
        print(f"seed={row['seed']}：没有剩余验证题，不报告正确率。请另建独立验证集。")
        continue
    print(f"seed={row['seed']}：正确 {row['correct']}/{row['total']}，"
          f"accuracy={row['accuracy']:.1%}，空回答={row['empty']}，"
          f"非法特殊token={row['invalid_special']}")
    print("第一题抽样回答：", repr(row["records"][0]["answer"]))


### 10.1 多轮聊天也是 token 拼接与上下文预算

消息结构为 `[{"role":"user","content":"你好"}, ...]`，最后一条必须是 user。先完整编码历史消息与当前问题，再追加 ASSISTANT 标记，模型随后续写回答。

不能无限追加历史：先为 `max_new_tokens` 预留空间，超过预算时删除最早的**完整 user/assistant 轮次**，尽量保留最近完整轮次。若 system + 当前 user 本身已过长，直接提示缩短；不把一个汉字的部分字节或半个角色标记当作完整问题。代码返回被保留的消息供自检，`chat_once` 本身只返回回答，不偷偷修改调用者的历史。

本教程只调用一次示例，不默认运行无限 `input()` 循环。smoke 输出乱码或空字符串很正常；一个从头开始、只训练几步的模型还没有学会中文对话。


In [ ]:
def build_chat_prompt(messages, max_new_tokens=32):
    if not messages or messages[-1].get("role") != "user":
        raise ValueError("聊天消息最后一条必须是user")
    if not 1 <= int(max_new_tokens) < model.cfg.max_seq_len:
        raise ValueError("回答长度必须大于0并小于上下文长度")
    copied = [{"role": message["role"], "content": str(message["content"])} for message in messages]
    system_messages = copied[:1] if copied[0]["role"] == "system" else []
    middle = copied[len(system_messages):-1]
    if len(middle) % 2 or any(middle[i]["role"] != ("user" if i % 2 == 0 else "assistant")
                              for i in range(len(middle))):
        raise ValueError("历史需要由完整user/assistant轮次组成，system只允许位于开头")
    rounds = [middle[i:i + 2] for i in range(0, len(middle), 2)]
    current_user = [copied[-1]]

    def encode_selected(selected):
        # SFT函数要求以assistant结束；空占位消息仅用于构造相同格式的前缀。
        ids, _ = encode_sft(selected + [{"role": "assistant", "content": ""}])
        # 删除空回答的EOS，使prompt恰好停在ASSISTANT标记后。
        return list(ids[:-1])

    budget = model.cfg.max_seq_len - int(max_new_tokens)
    minimal = system_messages + current_user
    if len(encode_selected(minimal)) > budget:
        raise ValueError("system与当前问题过长；请缩短文本或减少max_new_tokens")
    selected = system_messages + [message for pair in rounds for message in pair] + current_user
    while len(encode_selected(selected)) > budget and rounds:
        rounds.pop(0)
        selected = system_messages + [message for pair in rounds for message in pair] + current_user
    prompt_ids = encode_selected(selected)
    assert len(prompt_ids) + int(max_new_tokens) <= model.cfg.max_seq_len
    return prompt_ids, selected


def chat_once(messages, max_new_tokens=32):
    prompt_ids, _ = build_chat_prompt(messages, max_new_tokens)
    generated = generate(model, prompt_ids, max_new_tokens=max_new_tokens,
                         temperature=0.8, top_k=None, top_p=0.95, use_cache=True)
    completion = generated[len(prompt_ids):]
    if EOS_ID in completion:
        completion = completion[:completion.index(EOS_ID)]
    special_ids = {PAD_ID, BOS_ID, EOS_ID, USER_ID, ASSISTANT_ID, SYSTEM_ID}
    return tokenizer.decode([token for token in completion if token not in special_ids]).strip()


chat_messages = [{"role": "user", "content": "你好"},
                 {"role": "assistant", "content": "你好！"},
                 {"role": "user", "content": "什么是注意力？"}]
chat_prompt_ids, retained_messages = build_chat_prompt(chat_messages, max_new_tokens=16)
chat_answer = chat_once(chat_messages, max_new_tokens=16)
print("保留消息数：", len(retained_messages), "prompt长度：", len(chat_prompt_ids))
print("模型聊天答复：", repr(chat_answer))
print("提示：smoke文字混乱或空回答是短训练的预期现象。")
assert len(chat_prompt_ids) + 16 <= model.cfg.max_seq_len
assert len(chat_messages) == 3


### 10.2 KV cache 应该改变计算量，不能改变模型语义

不使用 cache 时，每生成一个 token 都重新计算整个前缀的 K/V。使用 cache 时，保留过去 K/V，只算新 token，并正确使用它的绝对位置。先比较“同一前缀、同一新 token”的最后 logits，再比较 top-k=1 的输出序列；这比仅观察一次计时更能发现缓存位置与掩码错误。

预计：最大 logits 差异很小，两种生成输出一致。GPU 混合精度允许小数值差异；短序列的 cache 未必更快，Python 调度开销可能占主要部分。计时只是当前环境的一次观察，不是性能结论。练习：画出无 cache 与 cache 的 K/V shape 在每一步如何变化。


In [ ]:
cache_prompt = encode_prompt("你好")
cache_token = tokenizer.encode("好")[0]
cache_device = next(model.parameters()).device
model.eval()
with torch.no_grad(), amp_context():
    prefix_tensor = torch.tensor([cache_prompt], dtype=torch.long, device=cache_device)
    _, prefix_cache = model(prefix_tensor, use_cache=True)
    step_logits, _ = model(torch.tensor([[cache_token]], dtype=torch.long, device=cache_device),
                           use_cache=True, kv_cache=prefix_cache)
    full_logits, _ = model(torch.tensor([cache_prompt + [cache_token]], dtype=torch.long,
                                       device=cache_device), use_cache=False)
logit_error = (step_logits[0, -1].float() - full_logits[0, -1].float()).abs().max().item()
torch.testing.assert_close(step_logits[0, -1].float(), full_logits[0, -1].float(),
                           atol=5e-2, rtol=1e-2)
print("缓存与完整前向的最大logits误差：", logit_error)


def timed_greedy(use_cache):
    if cache_device.type == "cuda":
        torch.cuda.synchronize(cache_device)
    started = time.perf_counter()
    # top_k=1保留唯一最高概率token，因此两次运行没有采样差异。
    generated = generate(model, cache_prompt, max_new_tokens=4 if PROFILE == "smoke" else 16,
                         temperature=1.0, top_k=1, top_p=1.0, use_cache=use_cache)
    if cache_device.type == "cuda":
        torch.cuda.synchronize(cache_device)
    return generated, time.perf_counter() - started


without_cache, seconds_without = timed_greedy(False)
with_cache, seconds_with = timed_greedy(True)
assert without_cache == with_cache
print(f"无缓存 {seconds_without:.4f}s；有缓存 {seconds_with:.4f}s；输出token一致")
del prefix_cache, step_logits, full_logits


### 10.3 保存、重载与继续练习

一个可重载的推理包至少有：权重 `state_dict`、模型配置、tokenizer 的 JSON，以及训练阶段说明。仅保存权重而丢失特殊 token ID/词表，会让同一整数序列被解码成不同文本。

下格保存当前模型，元数据写明 `PROFILE`、RL 尝试与实际更新数。重载函数创建一个**新变量**并返回，默认不调用，不覆盖当前训练好的 `model`。推理包不含优化器状态；精确续训还需保存优化器、调度器、数据进度与随机数状态。

自检：检查配置中的 `vocab_size` 等于重载 tokenizer 的词表大小；特殊 token ID 必须一致。权威材料可继续阅读 [DeepSeekMath](https://arxiv.org/abs/2402.03300)，并把本文的单次更新与论文完整实验区分开。


In [ ]:
BUNDLE_DIR = WORK_DIR / f"{PROFILE}_inference_bundle"
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
torch.save({name: value.detach().cpu() for name, value in model.state_dict().items()},
           BUNDLE_DIR / "model_state.pt")
(BUNDLE_DIR / "config.json").write_text(json.dumps(asdict(model.cfg), ensure_ascii=False, indent=2),
                                        encoding="utf-8")
# 底层Tokenizers直接保存JSON，保留byte-level规则、词表与特殊token约定。
tokenizer.backend.save(str(BUNDLE_DIR / "tokenizer.json"))
(BUNDLE_DIR / "training_stage.json").write_text(json.dumps({
    "profile": PROFILE, "stage": "rlvr_after_attempts", "rl_attempts": RL_STEPS,
    "rl_updates": rl_updates,
    "special_token_ids": {name: tokenizer.backend.token_to_id(name) for name in SPECIAL_TOKENS},
    "description": "从头训练的教学模型；短训练不代表对话或算术能力",
}, ensure_ascii=False, indent=2), encoding="utf-8")


def load_inference_bundle(bundle_dir, device="cpu"):
    bundle_dir = Path(bundle_dir)
    saved_config = json.loads((bundle_dir / "config.json").read_text(encoding="utf-8"))
    restored_tokenizer = TextTokenizer(Tokenizer.from_file(str(bundle_dir / "tokenizer.json")))
    saved_stage = json.loads((bundle_dir / "training_stage.json").read_text(encoding="utf-8"))
    for name, expected_id in saved_stage["special_token_ids"].items():
        if restored_tokenizer.backend.token_to_id(name) != expected_id:
            raise ValueError(f"特殊token {name} 的ID与保存时不同")
    restored_cfg = ModelConfig(**saved_config)
    if restored_cfg.vocab_size != restored_tokenizer.vocab_size:
        raise ValueError("配置与tokenizer的词表大小不一致")
    restored_model = TinyLLM(restored_cfg)
    state = torch.load(bundle_dir / "model_state.pt", map_location="cpu", weights_only=True)
    restored_model.load_state_dict(state, strict=True)
    restored_model.to(device).eval()
    return restored_model, restored_tokenizer, restored_cfg


print("已保存推理包：", BUNDLE_DIR.resolve())
print("文件：", sorted(path.name for path in BUNDLE_DIR.iterdir()))
# 需要重载时手动运行下一行，使用新变量且优先CPU，避免覆盖或显存双份占用。
# reloaded_model, reloaded_tokenizer, reloaded_cfg = load_inference_bundle(BUNDLE_DIR, device="cpu")


### 10.4 可选：用同一验证集比较 SFT 与 RL 后状态

比较必须用相同题目、相同采样种子、相同解码策略，以及同一设备与精度。下面开关默认为 False；打开后，把 SFT 检查点加载到临时 CPU 模型，同时将当前模型暂移 CPU，对两个状态都重新评估。完成后释放临时模型，并把当前模型移回原设备，不替换它的权重。full 的 CPU 评估会慢。

可能看到两阶段都是 0 分，也可能看到少量随机差别。这些小样本不能证明改进；尤其若 `rl_updates=0`，两阶段权重完全相同，同种子结果也应相同。

**下一步练习（按顺序）：**

1. 用自己的文字推导 `P-1` 为什么预测第一个回答 token，并画出 EOS 的位置。
2. 把正确答案概率实验改成负优势，验证概率下降；再把优势设成 0，验证不更新。
3. 在另一份数据副本中增加 SFT，直到模型确实能采到部分正确整数，再观察组内奖励差异。
4. 固定训练/验证/测试三份数据，增加种子与样本量，比较 SFT 和 RL；记录算力、更新数、奖励分布。
5. 若加入特殊 token 屏蔽或温度，先让 rollout 与 log probability 使用相同分布，再谈性能。

完整学习路线是：小张量弄清索引 → 跑通 smoke → 独立验证模型结构 → 增加数据与训练 → 用 heldout 判断能力。选择 full 只把模型规模提升到约 0.1B，不会自动增加数据质量或训练时长。


In [ ]:
RUN_STAGE_COMPARISON = False
if RUN_STAGE_COMPARISON:
    sft_checkpoint = torch.load(SFT_BEFORE_RL_PATH, map_location="cpu", weights_only=True)
    comparison_model = TinyLLM(ModelConfig(**sft_checkpoint["config"]))
    comparison_model.load_state_dict(sft_checkpoint["state_dict"], strict=True)
    comparison_model.eval()
    del sft_checkpoint
    original_device = next(model.parameters()).device
    original_training = model.training
    model.to("cpu").eval()
    try:
        sft_results = evaluate_arithmetic(comparison_model, HELDOUT_CASES)
        comparison_after = evaluate_arithmetic(model, HELDOUT_CASES)
        for before, after in zip(sft_results, comparison_after):
            print(f"seed={before['seed']}：SFT={before['correct']}/{before['total']}，"
                  f"RL后={after['correct']}/{after['total']}，实际RL更新={rl_updates}")
    finally:
        model.to(original_device).train(original_training)
    del comparison_model
    gc.collect()
else:
    print("阶段比较未运行；将RUN_STAGE_COMPARISON改为True后可执行同种子对照。")


### 10.5 验证集与测试集：不同用途的两次检查

训练时的 `val_tokens` 用于观察和调整设置；`test_tokens` 留到设置确定后作最终报告，不能看过结果后不断针对它调参还称为独立测试。下面对当前最终状态的模型计算一小份固定测试文本的 token 加权 loss 和 perplexity。

这不是纯预训练 checkpoint 的评估，因为模型已经历 SFT 和 RL 尝试。正式实验应分别重载各阶段的权重，用同样的测试文本比较。默认演示语料高度重复，所以这个数只说明整个评估流程能工作；真实语言能力仍需要独立、足量的文本与任务测试。

In [ ]:
# 不用随机 sampler；每次都评同一组均匀分布的文本窗口。
test_dataset = TokenWindowDataset(test_tokens, cfg.max_seq_len)
test_starts = np.linspace(0,len(test_dataset)-1,min(32,len(test_dataset)),dtype=np.int64).tolist()
test_loader = DataLoader(Subset(test_dataset,test_starts),batch_size=1,num_workers=0)
final_text_test = evaluate_lm(model,test_loader)
print("最终状态的固定测试文本子集:", final_text_test)
print("这是测试子集的指标；smoke模板语料不支持通用语言能力结论。")

## 11. 从教学小实验切换到真正的 0.1B 训练

你已经用同一套代码串起各阶段。现在把目标从“正确运行”变成“从真实数据学习”。**0.1B 指参数量，不是训练 token 数**。只切换配置而不增加数据，模型可能只是更容易记住小语料。

依次执行下面的步骤，每次只改变一个因素：

1. **准备真实文本。** 把 txt/jsonl 放在 `data/raw/`；优先限定语言与领域，检查字符编码、质量、重复和 train/val/test 的同源污染。JSONL 每行是 `{"text":"完整文档"}`。
2. **重启内核，改控制面板的 `PROFILE` 为 `full`。** 从头运行到模型定义，确认词表 8192、参数 98,324,736。不要把旧 smoke checkpoint 用在新模型上。
3. **先做 10～20 次 optimizer 更新。** 观察有限的 loss/梯度、显存、tokens/s、验证是否运行、checkpoint 是否可恢复，再考虑长训练。必要时用环境变量 `LLM_PT_STEPS` 设短步数；它是总 optimizer 更新目标。
4. **根据 token 预算设更新次数。** 保持模型配置、分词器、数据与有效 batch 固定，再增加训练步数。先完成一个足够小的里程碑，评估后继续。
5. **完成基座模型验证再做 SFT。** 自己的对话放在 `data/sft_train.jsonl`、`data/sft_val.jsonl`，每行 `{"messages":[{"role":"user","content":"问题"},{"role":"assistant","content":"回答"}]}`。第 8 章会读取这些文件，或使用合成教学数据。真实 SFT 要让格式、语言、答案质量与目标场景一致。
6. **只有任务已有成功样本时才认真尝试 RL。** 如果模型从不答对题，可验证奖励全为 0，RL 无从比较优劣；先改善数据/预训练/SFT。演示代码跳过这些组，并显示原因。

8 GiB 显存的起始配置：context 256、micro batch 1、梯度累计、SDPA、AMP、激活重计算。具体上限取决于软件与数据；出现 OOM 时先减 context/micro batch、清理不再使用的模型引用并重启内核，不要反复构建多个完整模型。

### 11.1 把“训练多久”变成可计算的预算

设 micro batch 为 $b$，梯度累计次数为 $a$，每条序列长度为 $T$，GPU 数量为 $g$：

$$\text{有效 batch}=bag,\qquad \text{预训练 token/更新}=bagT.$$

本教程只有一张 GPU，$g=1$。SFT 的有效监督 token 因回答长度与 padding 而变化，不能直接套固定 $T$ 的公式。

例如 $b=1,a=16,T=256$，每次预训练更新 4096 个 token；重复小数据也会计入这个数，**处理次数并不等于独立数据量**。计算预算时同时记下唯一语料 token 数、重复次数和处理 token 总数。

可先探索 10M、100M、1B 等逐级预算，并观察保留文本的验证损失和生成质量。研究中的约 `20×参数量` token 比例可作为一个历史计算参考，不能保证本模型、本中文词表或本数据达到某种质量；不同训练目标的最优预算也会变化。来源：[Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556)。

In [ ]:
# 这里只算预算，不触发训练。把这些数改成你的实际配置。
budget_micro_batch, budget_accum, budget_context = 1, 16, 256
tokens_per_update = budget_micro_batch * budget_accum * budget_context
for target_tokens in [10_000_000, 100_000_000, 1_000_000_000, 2_000_000_000]:
    required_updates = math.ceil(target_tokens/tokens_per_update)
    print(f"{target_tokens:,} token -> {required_updates:,} optimizer 更新（每次 {tokens_per_update:,} token）")

# 示例吞吐是假设值；请换成自己训练日志中的实测 tokens/s。
assumed_tokens_per_second = 1000
print("假设每秒1000 token，100M token约需小时:", round(100_000_000/assumed_tokens_per_second/3600,1))
print("该时长只是算术示例，不是本机测速或训练承诺。")

### 11.2 显存与吞吐实验：确认完整模型能训练一步

下面是一个**主动开启**的小探针，默认关闭。它构建独立的 98.3M 模型，在随机 token 上做一次 AdamW 更新，观察 AMP 和激活重计算能否工作；不是语言能力评估，也不会替换你训练的模型。

若当前正在持有完整模型、优化器、reference 等，先重启内核并只运行到模型定义，或使用项目终端 `python tools/validate_notebook.py --full-probe`。后者验证后会释放探针。随机 token 的 loss 大约接近 $\log V$，并无学习语言的含义。

In [ ]:
RUN_FULL_DEVICE_PROBE = False  # 想实际检查完整模型的一步训练时改为 True
if RUN_FULL_DEVICE_PROBE:
    if DEVICE.type != "cuda":
        raise RuntimeError("完整模型资源探针请使用 CUDA GPU；CPU 可以继续学习 smoke。")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    probe_model = TinyLLM(FULL_CFG).to(DEVICE).train()
    probe_optimizer = torch.optim.AdamW(probe_model.parameters(),lr=1e-4,foreach=False)
    probe_scaler = torch.amp.GradScaler("cuda",enabled=AMP_DTYPE==torch.float16)
    probe_ids = torch.randint(0,FULL_CFG.vocab_size,(1,257),device=DEVICE)
    torch.cuda.synchronize()
    probe_started = time.perf_counter()
    with amp_context():
        probe_logits, _ = probe_model(probe_ids[:,:-1])
        probe_loss = F.cross_entropy(probe_logits.reshape(-1,FULL_CFG.vocab_size),probe_ids[:,1:].reshape(-1))
    probe_scaler.scale(probe_loss).backward()
    probe_scaler.unscale_(probe_optimizer)
    probe_norm = torch.nn.utils.clip_grad_norm_(probe_model.parameters(),1.0)
    assert torch.isfinite(probe_loss) and torch.isfinite(probe_norm)
    probe_scaler.step(probe_optimizer)
    probe_scaler.update()
    torch.cuda.synchronize()
    print("参数:", sum(p.numel() for p in probe_model.parameters()))
    print("单步秒数:", round(time.perf_counter()-probe_started,3), "峰值已分配GiB:", round(torch.cuda.max_memory_allocated()/2**30,3))
    del probe_model, probe_optimizer, probe_scaler, probe_ids, probe_logits, probe_loss, probe_norm
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("完整模型资源探针关闭；需要时手动开启。")

### 11.3 复习：把系统讲清楚，才算真正掌握

先不看代码，尝试用自己的话回答：

1. 一个 token ID 为什么不是一个固定语义向量？Tokenizer 与 Embedding 分别学什么？
2. Q/K/V 怎样从 `[B,T,d]` 变成多头 shape？为什么注意力分数除以 $\sqrt{d_h}$？
3. 哪个测试能发现偷看未来？为什么训练和生成的计算方式不同？
4. $98,324,736$ 中每一项来自哪一块？增大 context 为什么不增加参数？
5. 已 shift 的标签为什么不能在 loss 里再次 shift？SFT 第一段回答的第一个 token 由哪个位置预测？
6. 梯度累计、梯度裁剪和 AMP scaler 分别解决什么？更新次数与 micro batch 次数怎样区分？
7. 预训练与 SFT 的模型结构是否改变？只有损失下降，能说明模型能聊天吗？
8. 同一 prompt 的所有 reward 一样，为什么组内 advantage 没有信息？
9. RL 的 `old_logp` 为什么不能跟着当前参数求梯度？top-p 为什么影响概率比的解释？
10. KV Cache 应保存什么？缓存时 RoPE 位置偏移和因果 mask 为何容易出错？

然后完成三个小项目：

- **理解项目：** 在小数据上去掉 RoPE、减少层数、取消权重共享，一次改一项，比较参数量与验证损失。用相同数据、种子和 token 预算控制变量。
- **训练项目：** 用一个限定主题的真实语料训练基座模型，保留三组从未参与选模型的提示，逐级记录续写与验证损失。
- **对话项目：** 写 50 条高质量、互不重复的指令训练数据和一组独立测试问题；比较 SFT 前后回答格式，再加入可验证任务。这个规模用于学习数据作用，仍不能证明通用能力。

保存一份实验表：配置、数据摘要、token 预算、loss、评估题集合、答案样例。避免只保存“最漂亮的一条回答”。

## 12. 继续阅读：用原始资料对照已经写过的代码

| 已学概念 | 原始资料/官方文档 | 对照本教材 |
|---|---|---|
| BPE 训练与持久化 | [Tokenizers Quicktour](https://huggingface.co/docs/tokenizers/quicktour) | 自己的 train 语料学习 merge，而非加载模型权重 |
| Attention mask 与 SDPA | [PyTorch 2.9 SDPA](https://docs.pytorch.org/docs/2.9/generated/torch.nn.functional.scaled_dot_product_attention.html) | 手写路径、布尔 mask、缓存非方阵 |
| AMP 与 GradScaler | [PyTorch 2.9 AMP](https://docs.pytorch.org/docs/2.9/amp.html) | 先 unscale 再裁剪，FP16 与 BF16 的区别 |
| 旋转位置编码 | [RoFormer](https://arxiv.org/abs/2104.09864) | RoPE 的坐标配对与位置偏移 |
| 参数/数据/计算预算 | [Chinchilla](https://arxiv.org/abs/2203.15556) | token 预算只是参考，实际质量需要验证 |
| 组相对策略优化 | [DeepSeekMath](https://arxiv.org/abs/2402.03300) | 本教材是简化教学实现，之后再读原始 GRPO |
| 人类偏好强化学习 | [InstructGPT](https://arxiv.org/abs/2203.02155) | 更完整的 RLHF：偏好标注、奖励模型与策略优化 |
| 直接偏好优化 | [DPO](https://arxiv.org/abs/2305.18290) | 偏好学习的另一条路线，不能把它等同于在线 RL rollout |

教材代码优先保证机制可见、可验证。后续工程扩展可以加入分布式训练、更长 context、更丰富评估、推理服务和批量 KV Cache；每次扩展前保留本教材的因果性、mask、缓存一致性与 checkpoint 检查。